# Step 1 정식 학습 — 단일 파이프 라우팅 (GNARL + MaskablePPO)

**이 노트북은 "Run All" 로 처음부터 끝까지 돌아간다.** 새 세션은 CLAUDE.md §0 을 먼저 읽을 것.

G3 검증(`g3_colab.ipynb`, 동결)에서 확정된 구성을 그대로 쓴다.

| 항목 | 값 | 근거 |
|---|---|---|
| 정책 구조 | **진짜 GNARL** (`gnarl_mode=True`) | CLAUDE.md §16-10 — 같은 천장에 6배 빨리 도달 |
| h 클리핑 | **RL 정책 `none` / A\* 탐색 `manhattan`** | §16-10 — RL 에서는 min() 이 기울기를 80% 차단 |
| 턴 각도 | 180° (역방향만 금지) | §16-2 — 90° 로 조이면 데드락 대유행 |
| 엘보 청구 | ≤67.5°→45° / ≤112.5°→90° / >112.5°→90°+45° | §16-5 |
| 종단 처리 | 목표=terminated(Φ=0) / 데드락=terminated(Φ≠0) / 타임아웃=truncated | §16-5, G3-N3 |
| 격자 | **50×50×20** (2.5m×2.5m×1.0m @50mm) | 실규모 |

## ⚠ 먼저 알아야 할 것 (추측이 아니라 실측)

G3 는 **20×20×10** 에서 eval success **0.40~0.45** 가 천장이었다 (§16-2, ablation 60런).
이 노트북의 격자는 **12.5배 넓고 A\* 최적경로가 2배 길다**(31칸 vs 16칸).
**졸업 기준 0.90 은 현재 확인된 성능보다 훨씬 위에 있다.** 5M 스텝으로 도달한다는 근거는 없다.
→ 셀 12 의 autoresearch 와 CLAUDE.md §13 의 `G3-L1`(남은 설계 후보: 관측 설계)이 그래서 살아 있다.

## Colab 에서 돌리는 법

1. **런타임 > 런타임 유형 변경 > T4 GPU** (없어도 CPU 로 돈다 — 실측 861 steps/s)
2. **런타임 > 모두 실행 (Run All)**
3. 처음에 두 번 멈춘다: Drive 권한 승인(셀 1b) 한 번뿐이고, 나머지는 전부 자동이다.
   (wandb 는 키가 없으면 로그인 프롬프트 없이 offline 로 떨어진다)
4. 끝나면 셀 10 이 **졸업 판정 표**를 출력한다.

**세션이 끊기면 같은 노트북을 Run All 만 다시 하면 된다.** 셀 1b 가 Drive 를 붙이고,
셀 8 이 `checkpoint_step1_final.zip` 을 자동으로 찾아 **남은 스텝만** 이어서 학습한다
(재개용 체크포인트는 평가할 때마다 갱신되므로 최대 100K 스텝만 잃는다).

## 소요시간 (이 격자에서 실측 — 4코어 CPU, 8 envs)

| 단계 | 시간 |
|---|---|
| 시나리오 생성 548개 (k 제약 A\* 도달성 검증 포함) | **84초** (캐시되므로 재개 시 0초) |
| h_net 사전학습 3,000 배치 | **약 3초** |
| 평가 1회 (미학습 48개) | **5~14초** (실패 궤적이 길수록 느리다) |
| 학습 5M 스텝 | **약 1.7시간** (861 steps/s) + 평가 50회 ≈ 11분 |

`추측:` T4 에서는 환경 롤아웃(numpy)이 병목이라 크게 빨라지지 않는다. 2 vCPU 런타임이면 오히려 느릴 수 있다.

## ⚠ 1차 정식 학습이 실패한 이유 — **Colab 에서 다시 돌리기 전에 읽을 것** (2026-09-20 실측)

같은 노트북으로 50×50×20 에서 1.4M 스텝을 돌린 결과다. 원자료: `results/step1_run1_tau_collapse.json`.

| 스텝 | 100k | 300k | 500k | 800k | 1.1M | 1.4M |
|---|---|---|---|---|---|---|
| eval success | **0.271** | 0.083 | 0.042 | **0.000** | 0.000 | 0.021 |
| train success | 0.120 | 0.040 | 0.040 | 0.000 | 0.010 | 0.010 |
| train ep_rew_mean | -6.27 | -5.76 | -4.95 | -4.71 | -4.64 | -4.77 |
| entropy (선택지 ≥2) | 1.42 | 0.94 | 0.59 | 0.47 | 0.43 | 0.48 |
| max_prob | 0.49 | 0.67 | 0.79 | 0.84 | 0.85 | 0.84 |
| **τ** | 0.0416 | — | 0.0165 | — | — | **0.0117** |
| h MAE (kg) | 13.7 | 13.2 | 10.4 | 10.4 | 11.2 | 13.0 |
| 직진 행동 비율 | 0.849 | 0.883 | 0.888 | 0.904 | 0.906 | 0.908 |

**보상은 정상이다.** 의심부터 거둬야 한다: A\* 도달의 할인수익 **+0.115** vs 300스텝 배회 **-0.987**,
평가 24개 시나리오 **전부** 도달이 이긴다. 보상의 최적해는 정확히 "목표 도달" 이다.
그런데도 `ep_rew_mean` 은 오르고 success 는 떨어졌다 — **엘보를 안 쓰는 쪽으로 보상을 벌고 있었다**
(엘보 1개 = 직진 4.7칸 = -0.064). 직진 비율이 0.849 → 0.908 로 단조 상승한 게 그 증거다.

### 원인 1 — τ 붕괴

τ 는 학습 파라미터(`gnarl_learn_tau=True`)인데 PPO 가 **줄이는 방향으로만** 밀었다:
`0.05 → 0.0416(100k) → 0.0165(500k) → 0.0130(1M) → 0.0117(1.4M)`.
이웃 간 f 차이의 표준편차가 0.0303 이므로 **logit 폭이 0.61 → 2.6 으로 4배 뾰족**해졌다.
entropy 1.42 → 0.47, max_prob 0.49 → 0.84 가 같이 움직인다. `ent_coef=0.05` 로는 못 막았다.

### 원인 2 — h 드리프트

**RL 중 h_core 를 "남은 J" 에 묶어두는 항이 어디에도 없다.** 사전학습 직후 1.32kg 이던 MAE 가
학습이 시작되자 10~13kg 로 벌어진다. h 는 §6 에서 **행동을 직접 고르는** 신경망이므로,
f = g + ΔJ + h 의 순위가 그만큼 망가진다. (CLAUDE.md G3-N8 이 "진짜 GNARL 에서는 미검증" 으로
남겨둔 자리 — 이제 실측됐다.)

### 둘이 겹치면

**틀린 순위에 점점 더 강하게 확신하는 정책**이 된다. 이게 0.271 → 0.000 의 기전이다.
G3(20×20×10)에서 안 보였던 이유는 `추측:` 그 격자에서는 랜덤 성공률이 0.146 이라
도달 신호가 자주 들어와 h 순위를 계속 교정했기 때문으로 보인다 (이 격자는 0.021).

### Colab 에서 쓸 손잡이 (셀 3 `cfg`)

**한 번에 하나만 켠다.** 여럿 동시에 켜면 원인이 분리되지 않는다.

| 후보 | 설정 | 성격 |
|---|---|---|
| **D. h 완전 동결** ← **현재 기본값** | `cfg.gnarl_freeze_h = True` | 원인 2 를 제거한 대조군. **진단용** |
| C. h 워밍업 동결 | `cfg.gnarl_freeze_h=True` + `cfg.gnarl_unfreeze_at=300_000` | 초반만 막고 학습은 유지 |
| A. τ 하한 | `cfg.gnarl_tau_min = 0.05` | 원인 1 만 차단. 가장 덜 침습적 |
| B. τ 고정 | `cfg.gnarl_learn_tau = False` | A 와 비슷하되 τ 가 아예 안 움직인다 |

**D 로 먼저 돌리는 이유:** D 에서도 붕괴하면 범인은 τ(A/B)고, D 에서 붕괴가 멈추면 드리프트가
범인이니 C 로 옮겨 "학습을 유지하면서" 같은 효과가 나는지 본다.

> **`gnarl_freeze_h=True` 의 대가:** logits = -(g + ΔJ + h)/τ 에서 **h_core 가 logit 경로의 유일한 신경망**이다.
> 동결하면 정책에 남는 학습 파라미터는 **τ 하나**뿐이라, PPO 는 순위를 못 바꾸고 확신의 세기만 조절한다.
> 즉 "A\* 휴리스틱 그리디 + 온도" 에 가까워지고, 보상만으로 배운다는 CLAUDE.md §1 과는 거리가 생긴다.
> 드리프트 차단이 효과가 있는지 **먼저 확인하는 용도**로 쓰고, 확인되면 `gnarl_unfreeze_at` 이나
> 보조 회귀항 쪽으로 옮겨가는 것을 권한다.

> **측정 규칙(§11)을 지킬 것:** 위 손잡이의 효과는 0.1 미만 차이면 단일 런으로 판정하지 않는다.
> 300K × 4~6시드 + 산포로 재고, 평가 풀은 고정한다(`SEED+1` 로 독립 생성되므로 자동으로 고정된다).


## 셀 1. 환경 설치

In [ ]:
#@title 1. 설치
# torch 는 Colab 런타임에 이미 들어 있다. 설치 목록에 넣으면 CUDA 빌드가 다른 것으로
# 덮여 GPU 가 사라질 수 있으므로 **넣지 않는다**.
import importlib.metadata as _md, sys

def _ver(p):
    try:    return _md.version(p)
    except Exception: return None

_np_before = _ver("numpy")

!pip install -q "sb3-contrib>=2.4,<3" wandb optuna rich tensorboard

import importlib
importlib.invalidate_caches()          # 방금 설치한 버전이 보이도록
_np_after = _ver("numpy")
if _np_before and _np_after != _np_before and "numpy" in sys.modules:
    print(f"\n!! numpy 가 {_np_before} → {_np_after} 로 바뀌었다. "
          f"[런타임 > 세션 다시 시작] 후 Run All 을 다시 할 것 !!\n")

import torch, numpy, gymnasium, stable_baselines3, sb3_contrib
print("python", sys.version.split()[0], "| numpy", numpy.__version__, "| torch", torch.__version__)
print("gymnasium", gymnasium.__version__, "| sb3", stable_baselines3.__version__,
      "| sb3-contrib", sb3_contrib.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU 없음 — CPU 로 돈다 (느리지만 동작한다). "
          "[런타임 > 런타임 유형 변경 > T4 GPU] 권장")


## 셀 1b. 작업 디렉터리 — **Colab 세션이 끊겨도 살아남게**

5M 학습은 몇 시간짜리다. Colab 런타임은 그 사이에 끊길 수 있고, `/content` 는 그때 통째로 날아간다.
여기서 **Google Drive 를 붙이고 그 안으로 `chdir`** 해두면 체크포인트·시나리오 캐시·결과 파일이 전부 Drive 에 남는다.

- Run All 하면 Drive 권한 승인 팝업이 **한 번** 뜬다. 승인하면 나머지는 자동이다.
- 끊긴 뒤에는 **같은 노트북을 Run All 하면 그대로 이어서 학습한다** (셀 8 이 `checkpoint_step1_final.zip` 을 자동으로 찾는다).
- Drive 를 안 쓰려면 `USE_DRIVE` 를 끈다. 대신 런타임이 죽으면 학습 결과도 같이 사라진다.
- 쓰는 용량: 시나리오 캐시 약 **115MB** + 체크포인트 약 2MB × 십여 개.


In [ ]:
#@title 1b. 작업 디렉터리 / Drive
USE_DRIVE = True                                          #@param {type:"boolean"}
WORKDIR   = "/content/drive/MyDrive/pipe-routing-step1"   #@param {type:"string"}

import os, sys
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")          # ← 권한 승인 팝업 (한 번)
elif IN_COLAB:
    WORKDIR = "/content/pipe-routing-step1"   # 런타임이 죽으면 같이 사라진다
    print("Drive 미사용 — 런타임이 끊기면 체크포인트도 사라진다")
else:
    WORKDIR = os.getcwd()                     # Colab 이 아니면 현재 디렉터리를 그대로 쓴다

os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)
print("작업 디렉터리:", os.getcwd())

_found = sorted(f for f in os.listdir(".")
                if f.startswith(("checkpoint_step1", "g3_scen_", "step1_")))
if _found:
    print("\n이미 있는 파일 (이어서 학습하거나 캐시를 재사용한다):")
    for f in _found:
        print(f"  {f:<46}{os.path.getsize(f)/1e6:>7.1f} MB")
else:
    print("빈 디렉터리 — 처음부터 시작한다")


## 셀 2. wandb 초기화

API key 가 없으면 자동으로 offline 로 떨어진다 (학습은 그대로 진행된다).

In [ ]:
#@title 2. wandb
# 대화형 로그인 프롬프트는 절대 띄우지 않는다 — Run All 이 거기서 멈춰버린다.
# 온라인으로 남기려면 Colab 왼쪽 열쇠 아이콘(비밀)에 WANDB_API_KEY 를 넣어둘 것.
import os, time
USE_WANDB = True   #@param {type:"boolean"}
run = None

def _wandb_key():
    if os.environ.get("WANDB_API_KEY"):
        return os.environ["WANDB_API_KEY"]
    try:
        from google.colab import userdata
        return userdata.get("WANDB_API_KEY")
    except Exception:
        return None

if USE_WANDB:
    try:
        import wandb
        from wandb.integration.sb3 import WandbCallback
    except Exception as e:
        print("wandb 없음 → 로깅 비활성:", e); USE_WANDB = False

if USE_WANDB:
    _key = _wandb_key()
    if _key:
        os.environ["WANDB_API_KEY"] = _key
        mode = "online"
    else:
        mode = "offline"
    try:
        run = wandb.init(project="pipe-routing-v3", name="step1", mode=mode,
                         save_code=False, dir=os.getcwd())
        print("wandb:", mode, "|", getattr(run, "url", "(offline)"))
        if mode == "offline":
            print("  (온라인 기록을 원하면 Colab 비밀에 WANDB_API_KEY 를 넣고 이 셀만 다시 실행)")
    except Exception as e:
        print("wandb 초기화 실패 → 로깅 비활성:", e); USE_WANDB = False; run = None
else:
    print("wandb 비활성 — 셀 10 의 그래프로만 확인한다")


## 셀 3. 설정 + 물리 상수 + 행동 공간

`cfg` 하나가 이 노트북의 모든 동작을 결정한다. **재개하려면 `cfg.resume_from` 만 채우면 된다.**

In [ ]:
#@title 3. cfg / JIS 상수 / 27방향

import math, json, pickle, random, heapq
from dataclasses import dataclass, asdict, field
from typing import Optional, List, Dict, Tuple
import numpy as np

SEED = 20260919


@dataclass
class Step1Config:
    # ---- 격자: 실규모 50×50×20 (2.5m × 2.5m × 1.0m @ 50mm) ----
    NX: int = 50
    NY: int = 50
    NZ: int = 20
    CELL_M: float = 0.05

    # ---- 파이프 (CLAUDE.md §3) ----
    nominal: str = "100A"          # → k=6 (§5 표)
    k_override: Optional[int] = None

    # ---- 진짜 GNARL (§6, G3 §16-10 확정) ----
    gnarl_mode: bool = True
    h_clip_mode: str = "none"      # RL 정책의 logits 용 — min() 이 기울기를 80% 차단하므로 끈다
    # (h_clip_astar 삭제 — 어디서도 읽지 않는 죽은 설정이었다. A* 는 h_octile3d_J 를 쓴다:
    #  26-연결 엄밀 하한이라 A* 해가 J-최적이다. 맨해튼은 26-연결에서 상한이라 쓰면 기준선이
    #  느슨해진다. 실측: 평가 A* 상태 626개에서 비허용 위반 0건 — CLAUDE.md §24-1)
    gnarl_tau_init: float = 0.05   # 이웃 간 f 차이 표준편차 ≈ 0.064 → τ=1.0 이면 균등분포가 된다
    gnarl_learn_tau: bool = True

    # ---- τ 붕괴 / h 드리프트 대응 (1차 정식 학습 실측, CLAUDE.md §19) ----
    # 1차 런(50x50x20, 1.4M)은 success 0.271@100k → 0.00~0.04 로 붕괴했다. 원인 두 가지:
    #
    #  (1) τ 붕괴 — τ 는 학습 파라미터인데 PPO 가 **줄이는** 방향으로만 밀었다.
    #      실측 τ: 0.05(초기) → 0.0416(100k) → 0.0165(500k) → 0.0130(1M) → 0.0117(1.4M).
    #      f 차이의 표준편차가 0.0303 이므로 logit 폭은 0.61 → 2.6 (4배 뾰족).
    #      entropy 1.42 → 0.47, maxp 0.49 → 0.84 가 정확히 같이 움직였다.
    #      ent_coef=0.05 로도 못 막았다 (정책 기울기가 더 셌다).
    #      → 대응: gnarl_tau_min 으로 하한을 걸거나 gnarl_learn_tau=False 로 고정한다.
    #
    #  (2) h 드리프트 — RL 중 h_core 를 '남은 J' 에 묶어두는 항이 **어디에도 없다**.
    #      사전학습 1.32kg → 학습 중 10~13kg 로 벌어졌다. f=g+ΔJ+h 의 순위가 망가진다.
    #      → 대응: gnarl_freeze_h 로 사전학습 h 를 고정한다.
    #
    #  (1)+(2)가 합쳐지면 "틀린 순위에 더 강하게 확신하는" 정책이 된다.
    #  보상식 자체는 정상이다 — A* 도달의 할인수익 +0.115 vs 300스텝 배회 -0.987,
    #  24개 평가 시나리오 전부에서 도달이 이긴다 (§19-3 실측). 보상을 의심하지 말 것.
    gnarl_freeze_h: bool = False        # 진단 전용 손잡이 (§20-2: 켜면 eval 이 스텝과 무관하게 고정된다)
    gnarl_unfreeze_at: Optional[int] = None  # 정수면 그 스텝에서 해제 (워밍업 동결). None = 계속 동결
    gnarl_tau_min: float = 0.05         # τ 하한. 붕괴를 막지는 못하지만(§20-2) §21 바탕 설정이다

    # ---- h 순위 정칙항 (CLAUDE.md §21, 2026-09-22 채택) ----
    # 붕괴의 실체는 h 의 순위 손상 하나다 (§20-4). PPO 에는 h 를 '남은 최적 J' 로 되돌리는
    # 항이 없어 단조로 파괴한다. 순위만 붙들고 나머지(편향·스케일)는 RL 에 맡긴다.
    # 강도는 λ 가 아니라 **롤아웃당 스텝 수**다 — 별도 Adam 이라 손실에 λ 를 곱해도 거의 상쇄된다.
    h_rank_batches: int = 256           # 롤아웃당 순위 기울기 스텝. 0 = 꺼짐 (강도 손잡이)
                                        # 256 = §21-8 확정값. 8 은 2M 에서 붕괴한다. 0 이면 §20-3 처럼 붕괴한다
    h_rank_lr: float = 1e-4             # 순위 옵티마이저(Adam) lr
    h_rank_bs: int = 512                # 배치당 순위 쌍 개수
    h_rank_metrics: bool = True         # §21-5 계기판(top-1 · ρ · |Δh|) 기록
    # (b) 경로 밖 순위 라벨 (§28). True 면 A* 경로 위 라벨에 정책 방문 경로 밖 결정 시점 라벨을 합친 캐시를 쓴다.
    #   캐시는 results/offpath_labels.py (collect → label) 로 만든다 — 노트북이 직접 만들지 않는다
    h_rank_offpath: bool = False
    # C2 (§30): 순위 쌍 가중. "none" = 모든 쌍 같은 무게 (기존, 비트 단위 동일) / "cost" = 참 비용 차 t_j − t_i 로 가중
    h_rank_weight: str = "none"
    # 라벨량 sweep (§41): None 이면 학습 풀 전체의 순위 라벨. 정수 N 이면 **학습 풀 앞 N 개 시나리오**의 라벨 그룹만 쓴다
    # (순위 정칙항과 BC 가 같은 캐시를 쓰므로 둘 다 줄어든다 — 실규모에서 둘 다 같은 이웃 A* 라벨에서 나오기 때문).
    # RL 학습 풀(500)·평가 풀·계기판 라벨은 그대로다. 그룹의 시나리오는 목표 좌표로 찾는다.
    h_rank_n_scen: Optional[int] = None

    # ---- Layer 0 ----
    max_turn_deg: float = 180.0    # §4 원문: 역방향만 금지
    s0_full: bool = True
    deadlock_1ply: bool = False

    # ---- Layer 2 (PBS, §9) ----
    phi_type: str = "manhattan"
    phi_goal_ratio: float = 0.8
    phi_cong_ratio: float = 0.2
    gamma: float = 0.99
    fail_cost_mode: str = "const"
    fail_cost_const: float = 2.0
    stage: str = "1c"              # 1c = 도달 + 벤딩 + J (정식 목적함수)
    reach_goal_bonus: float = 10.0
    reach_step_cost: float = 0.1

    # ---- 에피소드 ----
    max_steps: int = 300           # A* 최적경로가 약 31칸 → 넉넉히
    invalid_penalty: float = 0.01
    max_invalid: int = 5

    # ---- 관측 (§8) ----
    ray_max: int = 6
    # obs[76:78] 종단 정렬 채널의 정규화 (§22-4 H1)
    #   "diag" : along·cross 를 격자 대각선(73.1칸)으로 나눈다 — 결정 구간이 0.1 이하로 압축된다
    #   "k"    : along = tanh(along/(k+1)), cross = min(cross/(k+1), 1)
    #            실패 순간의 45.9% 가 s<6 이고 56.1% 가 추가 6칸 이상 선회를 강요받는다 (§22-5).
    #            조준은 목표 k칸 이전에 끝나야 하므로 결정 구간(0~k+1)에 해상도를 몰아준다
    obs_align_norm: str = "k"
    # obs[78:80] 포착점 2채널 (§25 H3 — **마지막 관측 채널**, R-OBS §8)
    #   현재 진행 광선 위에서 "한 번 꺾으면 목표에 정확히 닿는" 가장 가까운 지점까지 u칸, 꺾은 뒤 t칸.
    #   §25-6 채택 (2026-09-23): J_ratio 2.80 → 1.46, 진입 구간 스텝 39.2 → 9.1, success 0.875 → 0.990.
    #   False 로 두면 OBS_DIM 78 — §23 이전 체크포인트(h1k 등)를 열 때만 끈다
    obs_capture: bool = True
    # obs[80:132] H2-b 방향별 원거리 블록 점유율 52채널 (§31-7 사전 점검 · §32 — R-OBS 1회 해제, H2-b 한정)
    #   26방향 × 고리 c∈{8,20}: 중심 pos+c·d, 한 변 8칸 정육면체 [중심−4, 중심+4) 의 막힌 칸 비율 (격자 밖=막힘).
    #   ⛔ 블록 8 · 고리 8·20 · 정의는 사전 점검에서 고정 — 바꾸면 재설계 (§8 해제 조건 2). False 면 80차원 그대로
    obs_blocks: bool = False
    # ---- 관종 (Step 3~, §35-4 안 B · §36-6 구배 규칙 G0 결정) — 과제 사양 채널이라 R-OBS 대상 아님 (§8) ----
    # obs[80] = is_gravity (압력관 0 · 중력관 1). 중력관이면 환경·A* 가 공유하는 마스크에서 오르막(dz=+1)을 지운다 (G0).
    #   → 관종이 마스크를 바꾸므로 과제 사양 채널 자격이 성립한다.
    # obs[81] = slope_run_counter — **G0 에서는 상수 0 (예약 칸)**. 실규모 격자(400×300×100, §7)에서 정량 구배(G1/G2)를 켤 때
    #   "마지막 하강 뒤 수평 주행 / L" 을 싣는다. 그때 차원을 늘리면 전이 사슬에 이관 단계가 하나 더 생기므로 지금 자리를 비워 둔다 (§36-6).
    # False 면 OBS_DIM 80 그대로 — Step 1 (압력관만) 체크포인트·캐시와 비트 단위 호환. Step 3 학습에서 켠다.
    # 요구: obs_capture=True (obs[80] 위치 고정) · obs_blocks=False (재현 전용이라 함께 켜지 않는다).
    # 확장 경로 (Step 8 대비): 관종 ID one-hot 이 아니라 마스크·J 를 바꾸는 **물리 속성**을 obs[82:] 에 뒤로 붙인다 —
    #   k/k_max · kg_per_m 정규화 · 엘보 kg 비 … 한 번에 하나, zero-padding 전이 (§8 전이 규약).
    obs_pipe_type: bool = False
    # 중력관 시나리오 비율 (0 = Step 1 풀 그대로). 중력관은 목표가 시작보다 높으면 시작·목표를 바꾼다 (§36-4 (a), 실현율 99.8%).
    #   0 이 아니면 관종 붙인 풀을 별도 캐시(이름에 _gf{비율}_{규칙})로 만든다. 비율 자체는 Step 3 사전 등록 때 정한다 (미결)
    gravity_frac: float = 0.0
    # 구배 규칙 (S3-N2). 지금 구현은 G0(역구배 금지)뿐 — G1/G2 는 실규모 격자 확장 시 (§36-6)
    slope_rule: str = "G0"
    # S1-B1 재현 전용 (§26-4·§27). True 면 √2 방향 12개 직진에 45° 엘보를 붙이던 버그를 그대로 재현한다.
    #   §27 이전 체크포인트(H3·h1k 등)를 재평가할 때만 켠다 — 그 체크포인트는 버그 관측으로 학습됐다
    legacy_dj_edge_bug: bool = False
    # C4 (§33) — A* 궤적 행동 복제(BC). MSE 사전학습 뒤 · RL 전에 h_core 를 CE 로 bc_batches 만큼 학습한다.
    #   데이터 = 순위 라벨 캐시의 학습 그룹(A* 경로 위 결정 시점), 목표 = argmin t (동점 균등), τ 0.05 고정.
    #   §1-1: Step 1~3 은 A* 가 J-최적이라 모방이 천장이 아니다. §33-6 채택 (J_ratio −0.125 ± 0.041) → 기본 3000.
    #   0 이면 꺼짐 = H3fix 와 동일 (H3fix 재현 · 평가 전용 스크립트는 0 으로 둔다)
    bc_batches: int = 3000
    bc_lr: float = 1e-3
    bc_bs: int = 256
    # ---- Step 전이 (§38-2 · S3-N3) ----
    # 이전 Step 체크포인트(sb3 .zip)에서 정책 가중치를 옮겨 **새로 학습**한다 (재개가 아니다 — 스텝·옵티마이저는 새로).
    # 관측이 늘었으면 첫 층을 zero-padding (§8) 하고, 모양이 안 맞는 키(GNARL 에서 쓰지 않는 mlp_extractor 첫 층)는 건너뛴다.
    # None 이면 꺼짐 (기본 — 동작 변화 없음). Step 3 R2 는 pretrain_h=False · bc_batches=0 과 함께 쓴다 (§29-3).
    init_from: Optional[str] = None

    # ---- 시나리오 ----
    n_train_scen: int = 500
    n_eval_scen: int = 48
    max_start_dirs: int = 2
    obstacle_fill: float = 0.12
    # ---- 장애물 모델 (§42 ①, 2026-09-27) — 실규모 장애물 모델은 사용자 결정 전까지 보류 (§40-6). 파라미터화만 해 둔다 ----
    # "boxes" = 무작위 상자 (지금까지와 같다). 상자 크기는 칸 단위 (min, max 포함). 기본값은 Step 1 과 비트 단위 동일
    obstacle_model: str = "boxes"
    obstacle_box_min: Tuple[int, int, int] = (2, 2, 1)
    obstacle_box_max: Tuple[int, int, int] = (4, 4, 4)
    # 상자 개수 상한. None = 자동 (채움 목표 / 평균 상자 부피 × 20, 최소 1000). 예전 코드는 1000 고정이라
    # 실규모(12M 칸)에서 채움 0.19% 로 멈췄다 (§40-0). 검증 격자에서는 상한이 한 번도 걸리지 않는다 (§42 ① 관문)
    obstacle_max_boxes: Optional[int] = None
    # ---- 시나리오 비트 패킹 (§42 ②) — 실규모 격자 1개가 bool 로 27 MB (free + pad) 라 548개면 14.9 GB 다 (§40-0) ----
    # True 면 free 격자를 비트(1/8)로만 들고, sc["free"] · sc["pad"] 는 요청 시 풀어 최근 pack_cache_n 개만 메모리에 둔다.
    # None = 자동 (격자 칸 수 > 200만 이면 켠다). 검증 격자에서는 꺼져 있다 → 지금까지와 비트 단위 동일
    pack_scenarios: Optional[bool] = None
    pack_cache_n: int = 16
    min_path_cells: int = 30       # 격자가 커졌으므로 함께 올린다

    # ---- 신경망 ----
    features_dim: int = 64
    hidden: int = 128
    pretrain_h: bool = True
    # 사전학습 배치 수 (에폭이 아니라 배치로 센다).
    # G3 의 500 은 20x20x10 기준값이다. 이 격자(50x50x20)는 라벨 상태가 31k 개라
    # 500 배치 = 4 에폭에 불과해 **h 가 맨해튼보다도 못하다** (실측 val MAE):
    #   500배치 5.19kg / 1500 2.10 / 3000 1.51 / 6000 1.14  (맨해튼 2.88, octile3d 3.50)
    # 비용은 3000 배치에 3초다. 졸업 기준(h MAE <= 2.0kg) 아래로 내려오는 지점을 쓴다.
    pretrain_batches: int = 3000

    # ---- PPO ----
    n_envs: int = 8
    n_steps: int = 512
    batch_size: int = 64
    n_epochs: int = 10
    learning_rate: float = 1e-4    # §20-3: 3e-4 는 붕괴가 4~5배 빠르다
    ent_coef: float = 0.05
    clip_range: float = 0.2
    total_timesteps: int = 5_000_000

    # ---- 평가 / 체크포인트 ----
    eval_every: int = 100_000
    checkpoint_every: int = 500_000
    resume_from: Optional[str] = None   # 예: "checkpoint_step1_final"  (확장자 없이)
    auto_resume: bool = True            # 작업 디렉터리에 checkpoint_step1_final.zip 이 있으면 자동 재개

    # ---- 졸업 기준 (§7) ----
    grad_success: float = 0.90
    # §7 (2026-09-23 개정) — 경로 품질은 **J_ratio** 로 본다 (Step 1 의 J 에 이미 엘보가 들어 있다).
    # J_ratio·elbow_ratio 는 **성공 에피소드 기준**이다. 전체 기준으로 두면 실패(기준선의 ~10배)가
    # success 기준과 이중 계상돼, success 0.90 에서 성공 경로 J_ratio ≤ 0.23 을 요구하게 된다 (§24-3).
    grad_j_ratio: float = 1.20          # 최소 요건 (성공 에피소드)
    grad_elbow_ratio: float = 2.0       # 신규 — Step 3 이후로 넘길 엘보 습관 제어 (성공 에피소드)
    target_j_ratio: float = 1.10        # 목표선 — 졸업 후 개선 대상. 판정에 쓰지 않는다
    # ---- 계기판 경보 (판정 아님) ----
    alarm_hard_gap: float = 0.30        # 어려움 3분위 success 가 전체보다 이만큼 낮으면 경보 (§24-4)
    alarm_top1: float = 0.40            # §23-4: top-1 < 0.40 이면 경보
    alarm_rho: float = -0.30            # §23-4: ρ < −0.30 이면 경보 (ρ 는 음수일 때만 정보가 있다)
    grad_deadlock: float = 0.05
    # h 지표는 졸업 기준에서 제외됐다 (§7, 2026-09-22) — 계기판으로만 본다 (§20-4)
    grad_h_mae_kg: float = 2.0          # 참고값 (판정에 쓰지 않는다)

cfg = Step1Config()
print(json.dumps(asdict(cfg), ensure_ascii=False, indent=1))

# CLAUDE.md §3 직관 중량 (JIS G3454 Sch40)
JIS_PIPE = {   # 호칭경: (OD mm, kg/m)
    "15A": (21.7, 1.27), "25A": (34.0, 2.57),  "50A": (60.5, 5.44),
    "65A": (76.3, 9.11), "100A": (114.3, 16.10), "150A": (165.2, 28.20),
    "200A": (216.3, 42.50), "300A": (318.5, 79.70), "400A": (406.4, 124.00),
    "500A": (508.0, 185.00),
}
# CLAUDE.md §3 관이음 중량 (JIS B2312, LR) — 표에 있는 구경만
JIS_FITTING = {  # 호칭경: (elbow90 kg, elbow45 kg, tee kg, gate valve kg, B/L m)
    "25A": (0.30, 0.17, 0.54, 2.0, 0.12),
    "65A": (1.80, 0.99, 3.24, 10.0, 0.20),
    "100A": (3.80, 2.09, 6.84, 20.0, 0.24),
    "200A": (18.0, 9.90, 32.4, 80.0, 0.42),
    "300A": (51.0, 28.1, 91.8, 210.0, 0.64),
    "500A": (195.0, 107.0, 351.0, 730.0, 1.05),
}
# CLAUDE.md §5 k값 (LR 90°, CELL=50mm 기준)
K_TABLE = {"15A": 2, "25A": 2, "50A": 3, "65A": 3, "100A": 6,
           "150A": 9, "200A": 12, "300A": 18, "500A": 30}

@dataclass(frozen=True)
class PipeSpec:
    nominal: str
    od_mm: float
    kg_per_m: float
    elbow90_kg: float
    elbow45_kg: float
    tee_kg: float
    valve_kg: float
    bl_m: float
    k: int

def make_spec(nominal: str) -> PipeSpec:
    od, kgm = JIS_PIPE[nominal]
    if nominal not in JIS_FITTING:
        raise KeyError(f"{nominal} 관이음 중량이 CLAUDE.md §3 표에 없음 (보간 금지: 규격표 조회 필요)")
    e90, e45, tee, val, bl = JIS_FITTING[nominal]
    return PipeSpec(nominal, od, kgm, e90, e45, tee, val, bl, K_TABLE[nominal])

SPEC = make_spec(cfg.nominal)
K = cfg.k_override if cfg.k_override is not None else SPEC.k    # 100A → 6 (요구사항)
if cfg.k_override:
    print(f"[ablation] k 를 {SPEC.k} → {K} 로 강제 (J 상수는 {cfg.nominal} 그대로)")
PAD = max(cfg.ray_max, K + 1) + 1   # 패딩 폭: lookahead 최대 오프셋(1+k), ray 사거리 커버

# CLAUDE.md §6 — J 정규화 (미적용 시 h(n) 학습 실패, G1-Q2)
DIAG_J = math.sqrt(cfg.NX**2 + cfg.NY**2 + cfg.NZ**2) * cfg.CELL_M * SPEC.kg_per_m
J_MAX = DIAG_J                  # §9 Φ 정규화도 동일 스케일 사용
SCALE = 1.0 / DIAG_J

print(SPEC)
print(f"k = {K} 칸 (필요 직진 {K*cfg.CELL_M*1000:.0f} mm), PAD = {PAD}")
print(f"DIAG_J = {DIAG_J:.3f} kg | 직진 1칸 = {cfg.CELL_M*SPEC.kg_per_m:.4f} kg "
      f"| 90도 엘보 = {SPEC.elbow90_kg:.2f} kg (= 직진 {SPEC.elbow90_kg/(cfg.CELL_M*SPEC.kg_per_m):.1f}칸)")

DIRS26 = [
    (dx, dy, dz)
    for dx in (-1, 0, 1) for dy in (-1, 0, 1) for dz in (-1, 0, 1)
    if not (dx == 0 and dy == 0 and dz == 0)
]
assert len(DIRS26) == 26
DIR_TO_IDX = {d: i for i, d in enumerate(DIRS26)}
OPP26 = {i: DIR_TO_IDX[(-d[0], -d[1], -d[2])] for i, d in enumerate(DIRS26)}
assert all(OPP26[OPP26[i]] == i for i in range(26))     # V3-A: 26개 모두 정확

STAY = 26
N_ACTIONS = 27

MOVE_DIST = np.array([math.sqrt(dx*dx + dy*dy + dz*dz) for (dx, dy, dz) in DIRS26], dtype=np.float64)
DIRS_ARR = np.array(DIRS26, dtype=np.int64)             # (26,3)

def step_cost_kg(dir_idx: int, kg_per_m: float, cell_m: float) -> float:
    return float(MOVE_DIST[dir_idx]) * cell_m * kg_per_m

# 방향 i -> j 의 사잇각(도). 가능한 값: 0, 45, 54.7, 60, 70.5, 90, 109.5, 120, 125.3, 135, 180
_unit = DIRS_ARR / np.linalg.norm(DIRS_ARR, axis=1, keepdims=True)
TURN_DEG = np.degrees(np.arccos(np.clip(_unit @ _unit.T, -1.0, 1.0)))   # (26,26)
# S1-B1 (§26-4): √2 방향은 내적이 0.9999999999999998 이라 대각이 ~1e-6° 로 남아 직진이 45° 엘보로 청구됐다.
# 자기 방향 사잇각은 정의상 0 이다. (move_J 는 dir_idx != cur_dir 로 판정해 원래부터 정상이었다)
# 버그 재현(cfg.legacy_dj_edge_bug)은 셀 4 첫머리에서 한다 — 러너가 셀 3 뒤에 cfg 를 바꾸므로 여기서 읽으면 늦다
np.fill_diagonal(TURN_DEG, 0.0)

# 엘보 종류 판정 (JIS B2312 는 90°/45° 두 종류뿐이다).
# 26방향 격자에서 실제로 나오는 편향각: 45, 54.7, 60, 70.5, 90, 109.5, 120, 125.3, 135.
#   θ ≤ 67.5   → 45° 엘보 1개
#   θ ≤ 112.5  → 90° 엘보 1개
#   θ >  112.5 → 90° + 45° (엘보 하나로 못 꺾는다. 급선회를 J 에서 제대로 청구한다)
ELBOW45_MAX_DEG = 67.5
ELBOW90_MAX_DEG = 112.5
def elbow_kg_for(turn_deg: float, spec: PipeSpec) -> float:
    if turn_deg <= 1e-9:
        return 0.0
    if turn_deg <= ELBOW45_MAX_DEG:
        return spec.elbow45_kg
    if turn_deg <= ELBOW90_MAX_DEG:
        return spec.elbow90_kg
    return spec.elbow90_kg + spec.elbow45_kg

# 현재방향 i 에서 방향 j 로 갈 때의 엘보 중량 (kg). 이웃 26개 ΔJ 를 한 번에 만들기 위한 표.
ELBOW_KG_MAT = np.array([[elbow_kg_for(float(TURN_DEG[i, j]), SPEC) for j in range(26)]
                         for i in range(26)], dtype=np.float64)
STEP_KG = MOVE_DIST * cfg.CELL_M * SPEC.kg_per_m          # (26,) 직진분 ΔJ
def move_J_all(cur_dir: int) -> np.ndarray:
    """현재 방향에서 26방향 각각으로 갈 때의 ΔJ (kg). move_J() 의 벡터화 버전."""
    return STEP_KG + ELBOW_KG_MAT[cur_dir]

# lookahead 오프셋: 방향 i 로 꺾었을 때 검사할 칸들 (턴 목적지 기준 t=1..k)
LOOK_OFFS = np.stack([np.stack([DIRS_ARR[i] * t for t in range(1, K + 1)]) for i in range(26)])  # (26,k,3)

# 26방향 자유 직진 길이 채널 (obs[50:76], §8 확장) — 오프셋 t=1..k+1
RUN_LEN_CAP = K + 1
RUN_OFFS = np.stack([np.stack([DIRS_ARR[i] * t for t in range(1, RUN_LEN_CAP + 1)])
                     for i in range(26)])                # (26, k+1, 3)

# SDF 센서 10개 (§8 obs[40:50]) — 6면 + 수평 4대각
RAY_DIRS = np.array([(1,0,0), (-1,0,0), (0,1,0), (0,-1,0), (0,0,1), (0,0,-1),
                     (1,1,0), (1,-1,0), (-1,1,0), (-1,-1,0)], dtype=np.int64)
RAY_OFFS = np.stack([np.stack([RAY_DIRS[i] * t for t in range(1, cfg.ray_max + 1)])
                     for i in range(len(RAY_DIRS))])      # (10, ray_max, 3)
FACE_DIRS = np.array([(1,0,0), (-1,0,0), (0,1,0), (0,-1,0), (0,0,1), (0,0,-1)], dtype=np.int64)

print("행동 수:", N_ACTIONS, "| 유효 사잇각:", sorted(set(np.round(TURN_DEG.ravel(), 1))))

## 셀 4. 시나리오 생성 (§11 — k 제약 하 도달성 검증 포함)

마스크·관측·A\* 를 정의하고, **k 제약 하에서 A\* 로 풀리는 시나리오만** 채택한다.
`k=0` A\* 도달성으로는 부족하다 (V1 발견). 캐시 파일명에 시나리오를 바꾸는 설정이 전부 들어간다.

In [ ]:
#@title 4. 마스크 / 관측 / A* / 시나리오
DIMS = np.array([cfg.NX - 1, cfg.NY - 1, cfg.NZ - 1], dtype=np.float64)
# S1-B1 재현 전용 (§27): §27 이전 체크포인트를 재평가할 때만. 이웃 ΔJ 표를 버그 그대로 다시 만든다.
# (셀 3 뒤에 cfg 를 바꾸는 러너·재판정 스크립트에서도 반영되도록 셀 4 에서 적용한다)
if cfg.legacy_dj_edge_bug:
    _td_bug = np.degrees(np.arccos(np.clip(_unit @ _unit.T, -1.0, 1.0)))      # 대각 ~1e-6° 가 남는 원래 계산
    ELBOW_KG_MAT = np.array([[elbow_kg_for(float(_td_bug[i, j]), SPEC) for j in range(26)]
                             for i in range(26)], dtype=np.float64)
    print("⚠ legacy_dj_edge_bug=True — S1-B1 버그 재현 모드 (§27, 재평가 전용)")
TAIL_DIM = 4 if cfg.obs_capture else 2     # obs[76:76+TAIL_DIM] 종단 정렬 2 (§23) [+ 포착점 2 (§25)]
BLOCK_RINGS, BLOCK_SIDE = (8, 20), 8      # H2-b (§32) — 사전 점검(§31-7)에서 고정한 값. 바꾸면 재설계 (§8)
BLOCK_DIM = 26 * len(BLOCK_RINGS) if cfg.obs_blocks else 0
PIPE_DIM = 2 if cfg.obs_pipe_type else 0  # obs[80] is_gravity · obs[81] slope_run_counter (§36-6)
if cfg.obs_pipe_type:
    assert cfg.obs_capture and not cfg.obs_blocks, "obs_pipe_type 은 obs[80:82] 고정 — obs_capture=True · obs_blocks=False 필요"
assert cfg.slope_rule == "G0", "구배 규칙은 G0 만 구현돼 있다 — G1/G2 는 실규모 격자 확장 시 (§36-6, S3-N2)"
assert cfg.gravity_frac == 0.0 or cfg.obs_pipe_type, \
    "중력관이 섞인 풀은 관종 채널이 필요하다 — 정책이 모르면 같은 관측에서 다른 마스크를 만난다 (§8 과제 사양 채널)"
OBS_DIM = 76 + TAIL_DIM + BLOCK_DIM + PIPE_DIM   # §8 의 50 + 직진여유(26) + 종단 정렬 2 [+ 포착점 2] [+ 블록 52] [+ 관종 2]

# 칸 단위 대각선 — obs[76:78] 정규화용 (J_MAX 는 kg 단위라 여기 쓰면 스케일이 안 맞는다)
DIAG_CELLS = float(math.sqrt(cfg.NX**2 + cfg.NY**2 + cfg.NZ**2))

def align_channels(positions, dirs, goal, k=K) -> np.ndarray:
    """obs[76:78] — 목표 상대벡터를 **현재 진행 방향 기준으로** 분해한다 (§22-4 H1).

      along = 앞으로 얼마 남았는가 (음수면 이미 지나쳤다)
      cross = 지금 방향으로 직진하면 목표를 **몇 칸 빗나가는가**  ← 착지 실패의 핵심량

    실패의 98% 가 목표 2~3칸 앞에서의 착지 실패이고(§22-1), 빗나감은 위치·목표·방향의
    평행성 판정(비선형)이라 h_core 가 이웃마다 새로 배워야 했다.

    정규화는 cfg.obs_align_norm 이 고른다 (§22-5):
      "k"    — 결정 구간(0~k+1)에 해상도를 몰아준다. 조준은 목표 k칸 이전에 끝나야 한다
      "diag" — 격자 대각선 기준. 결정 구간이 범위의 0.1 이하로 압축된다
    """
    p = np.atleast_2d(np.asarray(positions, dtype=np.float64))
    di = np.atleast_1d(np.asarray(dirs, dtype=np.int64))
    g = np.atleast_2d(np.asarray(goal, dtype=np.float64))            # (1,3) 또는 (N,3)
    v = g - p                                                        # (N,3)
    du = DIRS_ARR[di] / MOVE_DIST[di][:, None]
    al = (v * du).sum(axis=1)
    cr = np.linalg.norm(v - al[:, None] * du, axis=1)
    out = np.empty((len(p), 2), dtype=np.float32)
    if cfg.obs_align_norm == "k":
        sc = float(k + 1) if k > 0 else 1.0
        out[:, 0] = np.tanh(al / sc)
        out[:, 1] = np.minimum(cr / sc, 1.0)
    else:
        out[:, 0] = np.clip(al / DIAG_CELLS, -1.0, 1.0)
        out[:, 1] = np.minimum(cr / DIAG_CELLS, 1.0)
    return out

CAP_HORIZON = 3 * (K + 1)          # 포착점 탐색 지평선 (칸). k=6 이면 21

def capture_channels(positions, dirs, ss, goal, k=K) -> np.ndarray:
    """obs[78:80] — 포착점 (§25 H3). 장애물·경계를 무시하는 순수 기하 (along/cross 와 같다).

    현재 방향 d 의 광선 위 p_u = pos + u·d 에서 w = goal − p_u 가
      · 0 이면                      → 포착 (t = 0). 직진 도달은 선회가 아니므로 s 조건이 없다
      · 격자방향 d' 과 평행 (d' ≠ ±d) 이고 u ≥ max(0, k−s) 이면 → 포착 (t = w 를 d' 로 가는 칸 수)
    가장 가까운 u 를 고른다. d' = d 는 '목표가 광선 위' 라는 뜻이라 w = 0 쪽에서 잡힌다.
      cap_u = tanh(u/(k+1))  (없으면 1)      cap_t = min(t/(k+1), 1)  (없으면 1)
    k+1 스케일은 §23-2 의 교훈이다. 이 채널을 바꿔 다시 재는 것은 R-OBS 로 금지돼 있다.
    """
    p = np.atleast_2d(np.asarray(positions, dtype=np.int64))
    di = np.atleast_1d(np.asarray(dirs, dtype=np.int64))
    s = np.atleast_1d(np.asarray(ss, dtype=np.int64))
    g = np.atleast_2d(np.asarray(goal, dtype=np.int64))
    d = DIRS_ARR[di].astype(np.int64)                                # (N,3)
    u = np.arange(CAP_HORIZON + 1)                                   # (U,)
    w = g[:, None, :] - (p[:, None, :] + u[None, :, None] * d[:, None, :])   # (N,U,3)
    a = np.abs(w); m = a.max(axis=2)                                 # m = t (체비쇼프 길이)
    par = ((a == m[..., None]) | (a == 0)).all(axis=2) & (m > 0)     # 격자방향과 평행
    sg = np.sign(w)
    same = (sg == d[:, None, :]).all(axis=2)
    opp = (sg == -d[:, None, :]).all(axis=2)
    u0 = np.maximum(0, k - np.minimum(s, k))
    turn = par & ~same & ~opp & (u[None, :] >= u0[:, None])
    cap = turn | (m == 0)
    hit = cap.any(axis=1)
    iu = cap.argmax(axis=1)
    tt = m[np.arange(len(p)), iu]                                    # w = 0 이면 t = 0
    sc = float(k + 1) if k > 0 else 1.0
    out = np.ones((len(p), 2), dtype=np.float32)
    out[hit, 0] = np.tanh(iu[hit] / sc)
    out[hit, 1] = np.minimum(tt[hit] / sc, 1.0)
    return out

def tail_channels(positions, dirs, ss, goal, k=K) -> np.ndarray:
    """obs[76:OBS_DIM] — (위치·방향·직진칸수·목표) 로만 정해지는 기하 채널. 캐시 변환도 이것을 쓴다."""
    parts = [align_channels(positions, dirs, goal, k)]
    if cfg.obs_capture:
        parts.append(capture_channels(positions, dirs, ss, goal, k))
    return np.concatenate(parts, axis=1)

_BLOCK_SAT = {}   # id(pad) -> (pad, 막힌 칸 3D 누적합). pad 는 시나리오 풀이 들고 있어 수명 동안 유지된다

def block_channels(pad, positions) -> np.ndarray:
    """obs[80:132] — H2-b 방향별 원거리 블록 점유율 (§32). 26방향 × 고리 (8, 20):
    중심 pos + c·d 의 한 변 8칸 정육면체 [중심−4, 중심+4) 안 막힌 칸 비율. 격자 밖 = 막힘 (§31-7 정의 그대로)."""
    hit = _BLOCK_SAT.get(id(pad))
    if hit is None or hit[0] is not pad:
        blk = ~pad[PAD:PAD + cfg.NX, PAD:PAD + cfg.NY, PAD:PAD + cfg.NZ]
        S = np.zeros((cfg.NX + 1, cfg.NY + 1, cfg.NZ + 1), np.int32)
        S[1:, 1:, 1:] = blk.astype(np.int32).cumsum(0).cumsum(1).cumsum(2)
        hit = _BLOCK_SAT[id(pad)] = (pad, S)
    S = hit[1]
    p = np.atleast_2d(np.asarray(positions, dtype=np.int64))
    hi_lim = np.array([cfg.NX, cfg.NY, cfg.NZ])
    out = []
    for c in BLOCK_RINGS:
        lo = p[:, None, :] + DIRS_ARR[None].astype(np.int64) * c - BLOCK_SIDE // 2      # (N,26,3)
        hi = lo + BLOCK_SIDE
        lo_c = np.clip(lo, 0, hi_lim); hi_c = np.clip(hi, 0, hi_lim)
        x0, y0, z0 = lo_c[..., 0], lo_c[..., 1], lo_c[..., 2]; x1, y1, z1 = hi_c[..., 0], hi_c[..., 1], hi_c[..., 2]
        inside = (S[x1, y1, z1] - S[x0, y1, z1] - S[x1, y0, z1] - S[x1, y1, z0]
                  + S[x0, y0, z1] + S[x0, y1, z0] + S[x1, y0, z0] - S[x0, y0, z0])
        vol_in = np.prod(np.maximum(hi_c - lo_c, 0), axis=-1)
        out.append((inside + (BLOCK_SIDE ** 3 - vol_in)) / BLOCK_SIDE ** 3)            # 격자 밖 칸 = 막힘
    return np.concatenate(out, axis=1).astype(np.float32)

def make_pad(free: np.ndarray) -> np.ndarray:
    """경계 밖 = 막힘(False) 이 되도록 PAD 만큼 패딩. free3() 의 경계검사를 인덱싱으로 대체."""
    p = np.zeros((cfg.NX + 2*PAD, cfg.NY + 2*PAD, cfg.NZ + 2*PAD), dtype=bool)
    p[PAD:PAD+cfg.NX, PAD:PAD+cfg.NY, PAD:PAD+cfg.NZ] = free
    return p

def free3(pad: np.ndarray, x: int, y: int, z: int) -> bool:
    if abs(x) > cfg.NX + PAD or abs(y) > cfg.NY + PAD or abs(z) > cfg.NZ + PAD:
        return False
    return bool(pad[x + PAD, y + PAD, z + PAD])

UPHILL = DIRS_ARR[:, 2] > 0   # 오르막 9방향 (z 인덱스 증가 = 위)

def action_mask_27(pad, pos, dir_idx, s, goal, k=K, max_turn_deg=None,
                   deadlock_1ply=None, gravity=False) -> np.ndarray:
    """Layer 0 전부. 반환 (27,) bool. 환경과 A* 가 공유한다.
    gravity=True (중력관) 면 G0 역구배 금지 — 오르막 이동을 지운다 (§36-6)."""
    if max_turn_deg is None:
        max_turn_deg = cfg.max_turn_deg
    if deadlock_1ply is None:
        deadlock_1ply = cfg.deadlock_1ply
    m = np.zeros(N_ACTIONS, dtype=bool)

    # §11 체크리스트: 현재 위치가 경계 밖/막힘이면 전부 False
    px, py, pz = pos[0] + PAD, pos[1] + PAD, pos[2] + PAD
    if not (0 <= px < pad.shape[0] and 0 <= py < pad.shape[1] and 0 <= pz < pad.shape[2]):
        return m
    if not pad[px, py, pz]:
        return m

    pos_a = np.asarray(pos, dtype=np.int64)
    goal_a = np.asarray(goal, dtype=np.int64)
    nb = pos_a + DIRS_ARR                                    # (26,3) 이동 목적지
    nbp = nb + PAD
    cand = pad[nbp[:, 0], nbp[:, 1], nbp[:, 2]].copy()       # 충돌 + 경계

    cand[OPP26[dir_idx]] = False                             # 역방향 (V3-A)
    if gravity:
        cand &= ~UPHILL                                      # 역구배 금지 (G0, 중력관 — §36-6)
    is_turn = np.arange(26) != dir_idx
    cand &= ~(is_turn & (TURN_DEG[dir_idx] > max_turn_deg))  # 급선회 상한 (미결 G3-N2)
    if k > 0 and s < k:
        cand &= ~is_turn                                     # 직진 칸수 부족 → 턴 금지

    # k<=0 (stage "1a") 이면 벤딩 반경 제약 자체가 없다 → 직진칸수/lookahead 둘 다 건너뛴다
    need = np.where(cand & is_turn)[0] if k > 0 else np.empty(0, dtype=np.int64)
    if need.size:
        cells = nb[need][:, None, :] + LOOK_OFFS[need]       # (n,k,3)
        cp = cells + PAD
        free_c = pad[cp[..., 0], cp[..., 1], cp[..., 2]]     # (n,k)
        blocked = ~free_c
        is_goal_c = (cells == goal_a).all(-1)                # (n,k)
        first_block = np.where(blocked.any(1), blocked.argmax(1), k)
        first_goal = np.where(is_goal_c.any(1), is_goal_c.argmax(1), k + 1)
        goal_is_next = (nb[need] == goal_a).all(1)
        ok = goal_is_next | (~blocked.any(1)) | (first_goal < first_block)
        cand[need[~ok]] = False

    # (선택) 1-ply 데드락 회피: 다음 상태에서 유효 이동이 0개인 행동도 잘라낸다.
    # 보상이 아니라 mask 이므로 Layer 0 규칙을 지킨다. 재귀 방지를 위해 내부 호출은 항상 off.
    if deadlock_1ply:
        for i in np.where(cand)[0]:
            i = int(i)
            npos = (pos_a[0] + DIRS_ARR[i, 0], pos_a[1] + DIRS_ARR[i, 1], pos_a[2] + DIRS_ARR[i, 2])
            if tuple(int(v) for v in npos) == tuple(int(v) for v in goal_a):
                continue
            ns = min(s + 1, k) if i == dir_idx else 1
            nxt = action_mask_27(pad, npos, i, ns, goal, k, max_turn_deg, deadlock_1ply=False,
                                 gravity=gravity)
            if not nxt[:26].any():
                cand[i] = False

    m[:26] = cand
    if not m.any():
        m[STAY] = True      # 데드락: MaskablePPO 는 유효 행동 ≥1 을 요구한다
    return m

# ---- Layer 1: J (kg) ----
def move_J(dir_idx: int, cur_dir: int, spec: PipeSpec) -> Tuple[float, float, int]:
    """한 칸 이동의 ΔJ. 반환 (ΔJ kg, 이동거리 m, 엘보종류 0=없음/45/90)."""
    dist_m = float(MOVE_DIST[dir_idx]) * cfg.CELL_M
    dJ = dist_m * spec.kg_per_m
    kind = 0
    if dir_idx != cur_dir:
        deg = float(TURN_DEG[cur_dir, dir_idx])
        dJ += elbow_kg_for(deg, spec)
        kind = 45 if deg <= ELBOW45_MAX_DEG else 90
        if deg > ELBOW90_MAX_DEG:
            kind = 135                        # 90 + 45 (아래에서 둘 다 집계)
    return dJ, dist_m, kind

# ---- 관측 (§8) ----
def build_obs(pad, pos, dir_idx, s, g_J, goal, k=K, gravity=False) -> np.ndarray:
    o = np.zeros(OBS_DIM, dtype=np.float32)
    pos_a = np.asarray(pos, dtype=np.float64)
    goal_a = np.asarray(goal, dtype=np.float64)
    o[0:3] = pos_a / DIMS                       # 현재 위치 xyz
    o[3:6] = goal_a / DIMS                      # 목표 위치 xyz
    o[6 + dir_idx] = 1.0                        # 현재 방향 one-hot (26)
    o[32] = (min(s, k) / k) if k > 0 else 0.0   # 직진 칸수 (k=0 이면 제약 자체가 없다)
    o[33] = min(g_J / J_MAX, 1.0)               # g_cost 누적 (정규화)
    ip = np.asarray(pos, dtype=np.int64)
    f = ip + FACE_DIRS + PAD                    # 이웃 6면 SDF
    o[34:40] = np.where(pad[f[:, 0], f[:, 1], f[:, 2]], 1.0, -1.0)
    rc = ip + RAY_OFFS + PAD                    # SDF 센서 10개
    rf = pad[rc[..., 0], rc[..., 1], rc[..., 2]]        # (10, ray_max)
    o[40:50] = np.cumprod(rf, axis=1).sum(axis=1) / cfg.ray_max
    # obs[50:76] — 26방향 자유 직진 길이 (k+1 칸에서 포화). §8 확장 채널.
    # 이게 없으면 에이전트는 '자기 진행방향의 벽'을 못 본다 (§8 아래 설명 참조).
    uc = ip + RUN_OFFS + PAD
    uf = pad[uc[..., 0], uc[..., 1], uc[..., 2]]        # (26, k+1)
    o[50:76] = np.cumprod(uf, axis=1).sum(axis=1) / RUN_LEN_CAP
    o[76:76 + TAIL_DIM] = tail_channels(pos_a, dir_idx, s, goal_a, k)[0]   # 종단 정렬 (§23) [+ 포착점 (§25)]
    if cfg.obs_blocks:
        o[76 + TAIL_DIM:] = block_channels(pad, pos_a[None].astype(np.int64))[0]   # H2-b (§32)
    if PIPE_DIM:
        o[80] = 1.0 if gravity else 0.0         # is_gravity (§36-6)
        o[81] = 0.0                             # slope_run_counter — G0 에서는 상수 0 (예약)
    return o

def build_obs_batch(pad, positions, dirs, ss, g_J, goal, k=K, gravity=False) -> np.ndarray:
    """build_obs() 를 N개 상태에 대해 한 번에. 이웃 27개를 매 스텝 인코딩하므로 벡터화가 필수다.
    (27번 따로 부르면 numpy 호출 오버헤드가 27배가 된다 — 여기서는 배열만 27배 커진다)"""
    positions = np.asarray(positions, dtype=np.int64)
    N = len(positions)
    o = np.zeros((N, OBS_DIM), dtype=np.float32)
    o[:, 0:3] = positions / DIMS
    o[:, 3:6] = np.asarray(goal, dtype=np.float64) / DIMS
    o[np.arange(N), 6 + np.asarray(dirs, dtype=np.int64)] = 1.0
    o[:, 32] = (np.minimum(np.asarray(ss), k) / k) if k > 0 else 0.0
    o[:, 33] = np.minimum(np.asarray(g_J, dtype=np.float64) / J_MAX, 1.0)
    ip = positions
    f = ip[:, None, :] + FACE_DIRS[None] + PAD                      # (N,6,3)
    o[:, 34:40] = np.where(pad[f[..., 0], f[..., 1], f[..., 2]], 1.0, -1.0)
    rc = ip[:, None, None, :] + RAY_OFFS[None] + PAD                # (N,10,R,3)
    rf = pad[rc[..., 0], rc[..., 1], rc[..., 2]]
    o[:, 40:50] = np.cumprod(rf, axis=2).sum(axis=2) / cfg.ray_max
    uc = ip[:, None, None, :] + RUN_OFFS[None] + PAD                # (N,26,k+1,3)
    uf = pad[uc[..., 0], uc[..., 1], uc[..., 2]]
    o[:, 50:76] = np.cumprod(uf, axis=2).sum(axis=2) / RUN_LEN_CAP
    o[:, 76:76 + TAIL_DIM] = tail_channels(positions, dirs, ss, goal, k)   # 종단 정렬 (§23) [+ 포착점 (§25)]
    if cfg.obs_blocks:
        o[:, 76 + TAIL_DIM:] = block_channels(pad, positions)                # H2-b (§32)
    if PIPE_DIM:
        o[:, 80] = 1.0 if gravity else 0.0      # is_gravity (§36-6)
        o[:, 81] = 0.0                          # slope_run_counter — G0 에서는 상수 0 (예약)
    return o

# 진짜 GNARL 모드의 관측: [자기 obs | 이웃 27개 obs | 이웃 27개 ΔJ(정규화)]
GNARL_OBS_DIM = OBS_DIM + N_ACTIONS * OBS_DIM + N_ACTIONS

def zero_pad_state_dict(sd: dict, new_in_dim: int,
                        first_layer_keys=("feat_net.0.weight", "h_net.0.weight",
                                          "h_core.0.weight", "v_core.0.weight")):
    """§8: Step 전이 시 관측 채널이 뒤에 붙으므로 입력 가중치를 0-padding 해서 load 호환."""
    out = dict(sd)
    for kkey in first_layer_keys:
        if kkey in out:
            w = out[kkey]
            if w.shape[1] < new_in_dim:
                pad_w = w.new_zeros((w.shape[0], new_in_dim - w.shape[1]))
                out[kkey] = torch.cat([w, pad_w], dim=1)
    return out

# ---- Layer 2: Φ (§9) ----
def Phi(pos, goal, pad=None, spec=SPEC) -> float:
    """PBS 포텐셜. 조건: Φ(종단)=0, state-only, γ_PBS=γ_PPO."""
    if tuple(pos) == tuple(goal):
        return 0.0                              # ← §9 조건 1 (종단 명시)
    if cfg.phi_type == "manhattan":
        d = abs(pos[0]-goal[0]) + abs(pos[1]-goal[1]) + abs(pos[2]-goal[2])
        return -d * cfg.CELL_M * spec.kg_per_m / J_MAX
    elif cfg.phi_type == "combined":
        e = math.dist(pos, goal)
        goal_term = -e * cfg.CELL_M * spec.kg_per_m / J_MAX
        ip = np.asarray(pos, dtype=np.int64)
        rc = ip + RAY_OFFS + PAD
        sdf = float(np.cumprod(pad[rc[..., 0], rc[..., 1], rc[..., 2]], axis=1).sum(axis=1).min())
        cong_term = -1.0 / (sdf * 5.0 + 1.0)
        return cfg.phi_goal_ratio * goal_term + cfg.phi_cong_ratio * cong_term
    raise ValueError(cfg.phi_type)

assert Phi((3, 3, 3), (3, 3, 3)) == 0.0, "§9 조건 1 위반"
print("Φ(start~) 예시:", round(Phi((0,0,0), (19,19,9)), 4), "| Φ(종단) =", Phi((5,5,5), (5,5,5)))

SQRT2, SQRT3 = math.sqrt(2.0), math.sqrt(3.0)

def h_octile3d_cells(pos, goal) -> float:
    """26-연결 격자에서 (pos→goal) 이동거리의 엄밀 하한 (칸 단위, 장애물 무시)."""
    d = sorted((abs(pos[0]-goal[0]), abs(pos[1]-goal[1]), abs(pos[2]-goal[2])), reverse=True)
    a, b, c = d
    return (a - b) * 1.0 + (b - c) * SQRT2 + c * SQRT3

# h_fn 규약: h_fn(pos, goal, dir_idx, s, pad) -> kg. 학습된 h(n) 도 같은 규약으로 끼운다.
def h_octile3d_J(pos, goal, dir_idx=0, s=0, pad=None, spec=SPEC) -> float:
    return h_octile3d_cells(pos, goal) * cfg.CELL_M * spec.kg_per_m

def h_manhattan_J(pos, goal, dir_idx=0, s=0, pad=None, spec=SPEC) -> float:
    d = abs(pos[0]-goal[0]) + abs(pos[1]-goal[1]) + abs(pos[2]-goal[2])
    return d * cfg.CELL_M * spec.kg_per_m

def astar_gnarl(pad, start, goal, start_dir, h_fn=h_octile3d_J, spec=SPEC, k=K,
                s0=None, max_expand=300_000, gravity=False):
    """CLAUDE.md §6 G1: g(n)=J, h(n)=h_fn. 상태 = (pos, dir, min(s,k)).
    gravity=True 면 환경과 같은 G0 마스크로 푼다 (기준선 공정성 §24-1). octile 휴리스틱은 마스크를 더해도 하한이다."""
    if s0 is None:
        s0 = k if cfg.s0_full else 0
    start = tuple(int(v) for v in start); goal = tuple(int(v) for v in goal)
    s_start = (start, int(start_dir), min(int(s0), k))
    g = {s_start: 0.0}
    came = {}
    cnt = 0
    heap = [(h_fn(start, goal, int(start_dir), min(int(s0), k), pad), 0, s_start)]
    expanded = 0
    closed = set()
    while heap:
        f, _, st = heapq.heappop(heap)
        if st in closed:
            continue
        closed.add(st)
        pos, d, s = st
        if pos == goal:
            return _reconstruct(came, st, g, spec, expanded)
        expanded += 1
        if expanded > max_expand:
            return None
        m = action_mask_27(pad, pos, d, s, goal, k, gravity=gravity)
        for i in np.where(m[:26])[0]:
            i = int(i)
            dJ, _, _ = move_J(i, d, spec)
            npos = (pos[0]+DIRS26[i][0], pos[1]+DIRS26[i][1], pos[2]+DIRS26[i][2])
            ns = min(s + 1, k) if i == d else 1
            nst = (npos, i, ns)
            ng = g[st] + dJ
            if ng < g.get(nst, math.inf) - 1e-12:
                g[nst] = ng
                came[nst] = st
                cnt += 1
                heapq.heappush(heap, (ng + h_fn(npos, goal, i, ns, pad), cnt, nst))
    return None

def _reconstruct(came, st, g, spec, expanded):
    states = [st]
    while st in came:
        st = came[st]
        states.append(st)
    states.reverse()
    J = g[states[-1]]
    len_m, b90, b45 = 0.0, 0, 0
    for a, b in zip(states[:-1], states[1:]):
        i, dprev = b[1], a[1]
        len_m += float(MOVE_DIST[i]) * cfg.CELL_M
        if i != dprev:
            deg = float(TURN_DEG[dprev, i])
            if deg <= ELBOW45_MAX_DEG: b45 += 1
            elif deg <= ELBOW90_MAX_DEG: b90 += 1
            else: b90 += 1; b45 += 1          # 급선회 = 90 + 45
    return dict(J=J, len_m=len_m, n_cells=len(states) - 1, bends90=b90, bends45=b45,
                expanded=expanded,
                states=[(s[0], s[1], s[2], g[s]) for s in states])   # h(n) 라벨용

from tqdm.auto import tqdm

def _obstacle_box_cap(target: int) -> int:
    if cfg.obstacle_max_boxes is not None:
        return int(cfg.obstacle_max_boxes)
    lo, hi = np.asarray(cfg.obstacle_box_min), np.asarray(cfg.obstacle_box_max)
    mean_vol = float(np.prod((lo + hi) / 2.0))
    return max(1000, int(math.ceil(20.0 * target / mean_vol)))

def gen_obstacle_grid(rng: np.random.Generator) -> np.ndarray:
    """장애물 격자 (True = 빈 칸). cfg.obstacle_model 로 모델을 고른다 — 지금은 "boxes" 하나 (§42 ①).
    boxes: 채움 목표(obstacle_fill)에 닿을 때까지 무작위 상자를 놓는다. 난수 호출 순서는 예전 코드와 같다."""
    assert cfg.obstacle_model == "boxes", f"장애물 모델 {cfg.obstacle_model!r} 은 아직 없다 (현장 자료 뒤 결정, §40-6)"
    free = np.ones((cfg.NX, cfg.NY, cfg.NZ), dtype=bool)
    target = int(cfg.obstacle_fill * free.size)
    cap = _obstacle_box_cap(target)
    (lx, ly, lz), (hx, hy, hz) = cfg.obstacle_box_min, cfg.obstacle_box_max
    filled = 0
    guard = 0
    while filled < target and guard < cap:
        guard += 1
        sx, sy, sz = (int(rng.integers(lx, hx + 1)), int(rng.integers(ly, hy + 1)), int(rng.integers(lz, hz + 1)))
        x = int(rng.integers(0, cfg.NX - sx + 1)); y = int(rng.integers(0, cfg.NY - sy + 1))
        z = int(rng.integers(0, cfg.NZ - sz + 1))
        blk = free[x:x+sx, y:y+sy, z:z+sz]
        filled += int(blk.sum())
        free[x:x+sx, y:y+sy, z:z+sz] = False
    return free

# ---- 시나리오 비트 패킹 (§42 ②) ----
from collections import OrderedDict

PACK_SCENARIOS = (cfg.pack_scenarios if cfg.pack_scenarios is not None
                  else cfg.NX * cfg.NY * cfg.NZ > 2_000_000)
_UNPACK_CACHE: "OrderedDict" = OrderedDict()

class PackedScenario(dict):
    """free 격자를 비트로만 저장한다. sc["free"] · sc["pad"] 는 없는 키라 __missing__ 이 풀어서 돌려준다 (저장하지 않는다).
    캐시 키는 비트 배열 객체 자체 — dict(sc, ...) 로 복사해도 같은 격자를 가리키고, 캐시가 비트 배열을 붙잡고 있어 id 재사용이 없다."""
    def __missing__(self, key):
        if key not in ("free", "pad") or "free_bits" not in self:
            raise KeyError(key)
        bits = dict.__getitem__(self, "free_bits")
        k = (id(bits), key)
        hit = _UNPACK_CACHE.get(k)
        if hit is not None and hit[0] is bits:
            _UNPACK_CACHE.move_to_end(k)
            return hit[1]
        shape = tuple(dict.__getitem__(self, "shape"))
        free = np.unpackbits(bits, count=int(np.prod(shape))).astype(bool).reshape(shape)
        arr = free if key == "free" else make_pad(free)
        _UNPACK_CACHE[k] = (bits, arr)
        while len(_UNPACK_CACHE) > max(2, cfg.pack_cache_n):
            _UNPACK_CACHE.popitem(last=False)
        return arr

def pack_scenario(sc: dict) -> "PackedScenario":
    if "free_bits" in sc:
        return sc if isinstance(sc, PackedScenario) else PackedScenario(sc)
    free = np.asarray(sc["free"], dtype=bool)
    out = PackedScenario({k: v for k, v in sc.items() if k not in ("free", "pad")})
    out["free_bits"] = np.packbits(free.ravel())
    out["shape"] = tuple(int(v) for v in free.shape)
    return out

def prepare_pool(pool: List[dict]) -> List[dict]:
    """캐시에서 읽은 뒤 · 관종 적용 뒤에 부른다. 패킹을 켜면 전부 PackedScenario 로, 끄면 그대로 (비트 형식 캐시는 풀어 쓴다)."""
    if PACK_SCENARIOS:
        return [pack_scenario(sc) for sc in pool]
    return [PackedScenario(sc) if ("free_bits" in sc and not isinstance(sc, PackedScenario)) else sc for sc in pool]

def plain_pool(pool: List[dict]) -> List[dict]:
    """pickle 용 — PackedScenario 는 노트북 네임스페이스의 클래스라 다른 프로세스에서 못 푼다. 일반 dict 로 저장한다."""
    return [dict(sc) for sc in pool]

def make_scenario(rng: np.random.Generator, max_tries: int = 60) -> Optional[dict]:
    for _ in range(max_tries):
        free = gen_obstacle_grid(rng)
        pad = make_pad(free)
        idx = np.argwhere(free)
        if len(idx) < 50:
            continue
        a = tuple(int(v) for v in idx[rng.integers(len(idx))])
        b = tuple(int(v) for v in idx[rng.integers(len(idx))])
        if sum(abs(x - y) for x, y in zip(a, b)) < cfg.min_path_cells:
            continue
        # 유효 초기 방향 후보 (첫 칸이 비어 있는 방향) → 셔플 후 최대 max_start_dirs 개
        cand = [i for i in range(26) if free3(pad, a[0]+DIRS26[i][0], a[1]+DIRS26[i][1], a[2]+DIRS26[i][2])]
        rng.shuffle(cand)
        dirs, sol = [], {}
        for d in cand[:max(1, cfg.max_start_dirs)]:
            r = astar_gnarl(pad, a, b, d)          # ← k 제약 하 도달성 검증
            if r is not None:
                dirs.append(int(d)); sol[int(d)] = r
        if not dirs:
            continue
        return dict(free=free, pad=pad, start=a, goal=b, dirs=dirs, astar=sol)
    return None

def build_pool(n: int, seed: int, desc: str) -> List[dict]:
    rng = np.random.default_rng(seed)
    pool = []
    pbar = tqdm(total=n, desc=desc)
    while len(pool) < n:
        sc = make_scenario(rng)
        if sc is not None:
            pool.append(pack_scenario(sc) if PACK_SCENARIOS else sc); pbar.update(1)
    pbar.close()
    return pool

# 캐시 키에 시나리오를 바꾸는 설정을 전부 넣는다.
# (구경/격자/장애물/턴상한/k 를 바꿨는데 옛 캐시가 조용히 로드되면 전부 무의미한 실험이 된다)
CACHE = (f"g3_scen_{cfg.nominal}_{cfg.NX}x{cfg.NY}x{cfg.NZ}_k{K}_"
         f"n{cfg.n_train_scen}+{cfg.n_eval_scen}_d{cfg.max_start_dirs}_"
         f"f{cfg.obstacle_fill}_t{cfg.max_turn_deg:.0f}_m{cfg.min_path_cells}"
         + ("" if (cfg.obstacle_model, tuple(cfg.obstacle_box_min), tuple(cfg.obstacle_box_max)) == ("boxes", (2, 2, 1), (4, 4, 4))
            else f"_{cfg.obstacle_model}{''.join(map(str, cfg.obstacle_box_min))}-{''.join(map(str, cfg.obstacle_box_max))}")
         + ".pkl")   # 장애물 모델이 기본과 다르면 캐시 이름을 가른다 (§11 캐시 키 원칙). 상자 개수 상한은 검증 격자에서 결과에 영향 없음
if os.path.exists(CACHE):
    with open(CACHE, "rb") as f:
        TRAIN_POOL, EVAL_POOL = pickle.load(f)
    TRAIN_POOL, EVAL_POOL = prepare_pool(TRAIN_POOL), prepare_pool(EVAL_POOL)
    print("캐시 로드:", CACHE)
else:
    t0 = time.time()
    TRAIN_POOL = build_pool(cfg.n_train_scen, SEED, "train scenarios")
    EVAL_POOL = build_pool(cfg.n_eval_scen, SEED + 1, "eval scenarios")
    with open(CACHE, "wb") as f:
        pickle.dump((plain_pool(TRAIN_POOL), plain_pool(EVAL_POOL)), f)
    print(f"생성 완료 {time.time()-t0:.1f}s → {CACHE}")

for _sc in TRAIN_POOL + EVAL_POOL:
    _sc.setdefault("gravity", False)   # 관종은 시나리오 속성이다 (§36-6). Step 1 캐시는 전부 압력관

def is_grav(sc) -> bool:
    return bool(sc.get("gravity", False))

def apply_pipe_types(pool: List[dict], frac: float, seed: int, desc: str) -> Tuple[List[dict], dict]:
    """풀의 frac 비율을 중력관으로 바꾼다 (§36-4 (a) — 흐름 뒤집기). 압력관 시나리오는 그대로 둔다.

    - 고를 시나리오: seed 로 정한 무작위 순열의 앞 round(frac·n) 개 (결정적)
    - 목표가 시작보다 높으면 시작·목표를 바꾼다 (중력관은 시작이 상류). 같은 층·아래면 그대로
    - 시작 방향: 바꾸지 않았으면 기존 dirs 를, 바꿨으면 생성기와 같은 규칙(첫 칸이 빈 방향 셔플 → 앞 max_start_dirs 개)으로 다시 고르고
      **G0 마스크 아래 k 제약 A\*** 로 도달성을 검증한다 (§11 체크리스트 [4]). 풀리는 방향이 없으면 압력관으로 남긴다
    """
    n = len(pool)
    pick = set(np.random.default_rng(seed).permutation(n)[:int(round(frac * n))].tolist())
    out, st = [], dict(picked=len(pick), gravity=0, swapped=0, failed=0)
    for idx, sc in enumerate(tqdm(pool, desc=desc)):
        if idx not in pick:
            out.append(sc); continue
        a, b = tuple(sc["start"]), tuple(sc["goal"])
        swapped = b[2] > a[2]
        if swapped:
            a, b = b, a
            cand = [i for i in range(26) if free3(sc["pad"], a[0]+DIRS26[i][0], a[1]+DIRS26[i][1], a[2]+DIRS26[i][2])]
            np.random.default_rng((seed, idx)).shuffle(cand)
            cand = cand[:max(1, cfg.max_start_dirs)]
        else:
            cand = list(sc["dirs"])
        dirs, sol = [], {}
        for d in cand:
            r = astar_gnarl(sc["pad"], a, b, d, gravity=True)
            if r is not None:
                dirs.append(int(d)); sol[int(d)] = r
        if not dirs:
            st["failed"] += 1; out.append(sc); continue
        out.append(dict(sc, start=a, goal=b, dirs=dirs, astar=sol, gravity=True, swapped=swapped))
        st["gravity"] += 1; st["swapped"] += int(swapped)
    return out, st

# 관종은 시나리오를 바꾸므로 캐시 키에 넣는다 (§11 캐시 키 원칙). 0 이면 Step 1 캐시 이름 그대로
if cfg.gravity_frac > 0:
    BASE_CACHE = CACHE
    CACHE = CACHE.replace(".pkl", f"_gf{cfg.gravity_frac:g}_{cfg.slope_rule}.pkl")
    if os.path.exists(CACHE):
        with open(CACHE, "rb") as f:
            TRAIN_POOL, EVAL_POOL, PIPE_TYPE_STATS = pickle.load(f)
        TRAIN_POOL, EVAL_POOL = prepare_pool(TRAIN_POOL), prepare_pool(EVAL_POOL)
        print("관종 캐시 로드:", CACHE)
    else:
        t0 = time.time()
        TRAIN_POOL, _st_tr = apply_pipe_types(TRAIN_POOL, cfg.gravity_frac, SEED + 10_000, "pipe types (train)")
        EVAL_POOL, _st_ev = apply_pipe_types(EVAL_POOL, cfg.gravity_frac, SEED + 10_001, "pipe types (eval)")
        PIPE_TYPE_STATS = dict(train=_st_tr, eval=_st_ev)
        TRAIN_POOL, EVAL_POOL = prepare_pool(TRAIN_POOL), prepare_pool(EVAL_POOL)   # dict(sc, ...) 복사가 일반 dict 로 만든다
        with open(CACHE, "wb") as f:
            pickle.dump((plain_pool(TRAIN_POOL), plain_pool(EVAL_POOL), PIPE_TYPE_STATS), f)
        print(f"관종 풀 생성 {time.time()-t0:.1f}s → {CACHE}")
    print(f"중력관: train {sum(map(is_grav, TRAIN_POOL))}/{len(TRAIN_POOL)} · eval {sum(map(is_grav, EVAL_POOL))}/{len(EVAL_POOL)} "
          f"| {PIPE_TYPE_STATS}")

_stat = [sc["astar"][sc["dirs"][0]] for sc in EVAL_POOL]
print(f"train={len(TRAIN_POOL)} eval={len(EVAL_POOL)}")
print(f"eval 기준선: J {np.mean([s['J'] for s in _stat]):.2f} kg | "
      f"길이 {np.mean([s['len_m'] for s in _stat]):.2f} m | "
      f"칸수 {np.mean([s['n_cells'] for s in _stat]):.1f} | "
      f"벤딩 {np.mean([s['bends90']+s['bends45'] for s in _stat]):.1f} | "
      f"A* 확장노드 {np.mean([s['expanded'] for s in _stat]):.0f}")

## 셀 5. 환경 + §11 체크리스트 10/10

**체크리스트가 하나라도 실패하면 assert 로 멈춘다.** 학습을 시작하지 않는 것이 맞다.

In [ ]:
#@title 5. 환경 + 체크리스트
import gymnasium as gym
from gymnasium import spaces

def fail_cost_norm(pos, goal) -> float:
    """데드락 종단 비용. '미완성 경로는 남은 구간 값을 안 낸 게 아니다'.

    Layer 0 벌점이 아니다 (하드 제약은 전부 mask 로 처리되어 애초에 선택 불가).
    이것은 Layer 1 목적함수의 종단 조건이다: 목표에 닿지 못한 경로의 J 는
    '쓴 J + 남은 최소 J' 이고, 중간에 끊었다고 깎아주지 않는다.
    성공 궤적끼리의 순위는 건드리지 않으므로 최적 정책은 그대로다.
    """
    if cfg.fail_cost_mode == "none":
        return 0.0
    if cfg.fail_cost_mode == "const":
        return cfg.fail_cost_const
    return h_octile3d_J(pos, goal) / J_MAX          # 26-연결 엄밀 하한

class PipeRoutingEnvStep1(gym.Env):
    metadata = {"render_modes": []}

    def __init__(self, pool: List[dict], spec: PipeSpec = SPEC, eval_mode: bool = False,
                 seed: Optional[int] = None):
        super().__init__()
        self.pool, self.spec, self.eval_mode = pool, spec, eval_mode
        # stage "1a" 는 벤딩 반경 제약이 없다. 단 **시나리오 풀은 k=6 으로 생성된 것을 그대로 쓴다**
        # (풀을 다시 뽑으면 평가셋이 달라져 1a→1b→1c 전이 비교와 기존 수치 비교가 전부 깨진다).
        self.k = 0 if cfg.stage == "1a" else (cfg.k_override if cfg.k_override is not None else spec.k)
        self.rng = np.random.default_rng(seed)
        self.action_space = spaces.Discrete(N_ACTIONS)
        dim = GNARL_OBS_DIM if cfg.gnarl_mode else OBS_DIM
        self.observation_space = spaces.Box(-1.0, 1.0, (dim,), dtype=np.float32)
        self._cursor = 0
        self.sc = None

    # ---------- helpers ----------
    def _obs(self):
        own = build_obs(self.pad, self.pos, self.dir, self.s, self.g_J, self.goal, self.k, gravity=self.gravity)
        if not cfg.gnarl_mode:
            return own
        # --- 이웃 27개를 인코딩해 관측에 실어 보낸다 (정책이 f=g+h 를 계산할 수 있도록) ---
        # 무효 이웃도 그대로 인코딩한다. 어차피 action mask 가 분포에서 제거하므로 값은 쓰이지 않는다.
        # (PAD 가 k+1 오프셋까지 덮으므로 격자 밖 이웃도 인덱싱은 안전하다)
        pos = np.asarray(self.pos, dtype=np.int64)
        npos = pos + DIRS_ARR                                   # (26,3)
        ndir = np.arange(26)
        ns = np.where(ndir == self.dir, min(self.s + 1, self.k) if self.k > 0 else 0, 1)
        dJ = move_J_all(self.dir)                               # (26,) kg
        nb = build_obs_batch(self.pad, npos, ndir, ns, self.g_J + dJ, self.goal, self.k, gravity=self.gravity)
        nbr = np.concatenate([nb, own[None, :]], axis=0)        # (27,OBS_DIM) — 마지막은 STAY(제자리)
        dJ27 = np.concatenate([dJ, [0.0]]) / J_MAX              # STAY 는 ΔJ=0
        return np.concatenate([own, nbr.ravel(), dJ27.astype(np.float32)]).astype(np.float32)

    def _owed_J(self) -> float:
        """미완성 경로가 **아직 빚진 J** — 지금 상태에서 목표까지의 k 제약 A* 최적 J (kg).

        §2 종단 원칙 (fail_cost_norm 주석): '미완성 경로의 J = 쓴 J + 남은 최소 J' —
        중간에 끊었다고 깎아주지 않는다. 이걸 빼고 '쓴 J' 만 세면 일찍 포기할수록 점수가 좋아진다.
        A* 해가 없으면(그 상태에서 도달 불가) 26-연결 엄밀 하한으로 대신한다 (과소 계상 쪽).
        """
        sol = astar_gnarl(self.pad, tuple(self.pos), tuple(int(v) for v in self.goal),
                          int(self.dir), s0=int(self.s), gravity=self.gravity)
        return float(sol["J"]) if sol is not None else float(h_octile3d_J(self.pos, self.goal))

    def _info(self, success=False, deadlock=False, timeout=False):
        base = self.base
        ok = success and base is not None
        bb = float(base["bends90"] + base["bends45"]) if base else float("nan")
        # J_ratio_all — 계기판 전용 (판정 아님, §7). 평가 모드의 종단에서만 계산한다 (학습엔 불필요).
        jr_all = float("nan")
        if base is not None and self.eval_mode and (success or deadlock or timeout):
            jr_all = float((self.g_J + (0.0 if success else self._owed_J())) / base["J"])
        return dict(
            success=float(success),
            deadlock=float(deadlock),
            timeout=float(timeout),
            j_agent=float(self.g_J),
            j_ratio=float(self.g_J / base["J"]) if ok else float("nan"),
            length_ratio=float(self.len_m / base["len_m"]) if ok else float("nan"),
            n_bends=float(self.b90 + self.b45),
            bends_base=bb,
            # 기준선 엘보가 0 이면 비율이 정의되지 않으므로 제외한다 (현재 평가 풀엔 없다: min 1, §24-2)
            elbow_ratio=float((self.b90 + self.b45) / bb) if (ok and bb > 0) else float("nan"),
            j_ratio_all=jr_all,
            n_invalid=float(self.n_invalid),
            ep_steps=float(self.t),
        )

    # ---------- gym API ----------
    def reset(self, *, seed=None, options=None):
        if seed is not None:
            self.rng = np.random.default_rng(seed)
        options = options or {}
        if "index" in options:
            i = int(options["index"]) % len(self.pool)
        elif self.eval_mode:
            i = self._cursor % len(self.pool); self._cursor += 1
        else:
            i = int(self.rng.integers(len(self.pool)))
        sc = self.pool[i]
        self.sc, self.pad = sc, sc["pad"]
        self.gravity = is_grav(sc)                               # 관종 — 마스크(G0)와 obs[80] 을 가른다 (§36-6)
        self.pos, self.goal = sc["start"], sc["goal"]

        # §11: 초기 방향 하드코딩 금지 — 유효 초기방향 집합에서 선택 (V3-D)
        if "dir" in options:
            self.dir = int(options["dir"])
        elif self.eval_mode:
            self.dir = int(sc["dirs"][0])                       # 평가는 결정적
        else:
            self.dir = int(self.rng.choice(sc["dirs"]))
        self.base = sc["astar"].get(self.dir)

        self.s = self.k if cfg.s0_full else 0                   # 시작 노즐 직관 가정
        self.g_J, self.len_m = 0.0, 0.0
        self.b90 = self.b45 = 0
        self.t, self.n_invalid = 0, 0
        self.path = [self.pos]
        return self._obs(), self._info()

    def action_masks(self) -> np.ndarray:
        # §11: 위치가 경계 밖이면 action_mask_27 이 전부 False 를 돌려준다
        return action_mask_27(self.pad, self.pos, self.dir, self.s, self.goal, self.k, gravity=self.gravity)

    def step(self, action: int):
        action = int(action)
        self.t += 1
        m = self.action_masks()
        reward = 0.0
        terminated = truncated = False

        # --- §11: 경계/유효성 검사. 무효 행동 = 제자리 유지 + 페널티 (G2-D 버그 원인) ---
        if action < 0 or action >= N_ACTIONS or not m[action]:
            self.n_invalid += 1
            reward -= cfg.invalid_penalty
            if self.n_invalid >= cfg.max_invalid or self.t >= cfg.max_steps:
                truncated = True
            return self._obs(), reward, terminated, truncated, self._info(timeout=truncated)

        if action == STAY:
            # 유효 이동이 0개일 때만 열리는 행동 = 데드락 (V1 의 2% 사례).
            # terminated=True: 이어질 상태가 실제로 없으므로 부트스트랩하면 허구의 미래가치가 붙는다.
            # Φ(데드락) 은 0 으로 만들지 않는다 (셀 8 설명 참조).
            if cfg.stage == "1c":
                reward -= fail_cost_norm(self.pos, self.goal)   # 미완성 잔여 J (1a/1b 는 보상식 고정)
            terminated = True
            return self._obs(), reward, terminated, truncated, self._info(deadlock=True)

        prev_pos = self.pos
        reach_mode = cfg.stage in ("1a", "1b")
        phi_prev = 0.0 if reach_mode else Phi(prev_pos, self.goal, self.pad, self.spec)

        dJ, dist_m, kind = move_J(action, self.dir, self.spec)          # Layer 1 (1a/1b 에서도 계측은 한다)
        self.pos = (prev_pos[0] + DIRS26[action][0],
                    prev_pos[1] + DIRS26[action][1],
                    prev_pos[2] + DIRS26[action][2])
        self.s = min(self.s + 1, self.k) if action == self.dir else 1
        self.dir = action
        self.g_J += dJ
        self.len_m += dist_m
        if kind == 90: self.b90 += 1
        elif kind == 45: self.b45 += 1
        elif kind == 135: self.b90 += 1; self.b45 += 1
        self.path.append(self.pos)

        if reach_mode:
            # 커리큘럼 1a/1b: 도달 +10 / 스텝 -0.1 만. J 도 PBS 도 보상에 넣지 않는다.
            reward += -cfg.reach_step_cost
        else:
            phi_new = Phi(self.pos, self.goal, self.pad, self.spec)      # Φ(목표)=0 보장
            reward += -dJ / J_MAX + cfg.gamma * phi_new - phi_prev       # Layer 2

        if self.pos == self.goal:
            if reach_mode:
                reward += cfg.reach_goal_bonus
            terminated = True                                            # 진짜 종단
            return self._obs(), reward, terminated, truncated, self._info(success=True)
        if self.t >= cfg.max_steps:
            truncated = True                                             # 부트스트랩 대상
            return self._obs(), reward, terminated, truncated, self._info(timeout=True)
        return self._obs(), reward, terminated, truncated, self._info()

INFO_KEYS = ("success", "deadlock", "timeout", "j_ratio", "length_ratio", "elbow_ratio",
             "n_bends", "ep_steps")

_e = PipeRoutingEnvStep1(EVAL_POOL, eval_mode=True)
_o, _i = _e.reset(options={"index": 0})
print("obs dim:", _o.shape, "| 유효 행동 수:", int(_e.action_masks().sum()),
      "| 기준선 J:", round(_e.base["J"], 2))

def run_checklist_tests(pool=EVAL_POOL, n_ep=40, seed=7):
    rng = np.random.default_rng(seed)
    env = PipeRoutingEnvStep1(pool, eval_mode=False, seed=seed)
    results = {}

    # [1] step() 경계 검사 + 무효 행동 = 제자리 유지 + 페널티 (G2-D)
    env.reset(options={"index": 0})
    bad = int(np.where(~env.action_masks())[0][0])
    p0 = env.pos
    _, r, term, trunc, info = env.step(bad)
    results["[1] 무효행동=제자리+페널티"] = (env.pos == p0 and r < 0 and info["n_invalid"] == 1.0)

    # [2] action_masks(): 경계 밖 위치면 전부 False
    sc = pool[0]
    out = action_mask_27(sc["pad"], (-3, 5, 5), 0, K, sc["goal"])
    out2 = action_mask_27(sc["pad"], (cfg.NX + 2, 5, 5), 0, K, sc["goal"])
    results["[2] 경계밖 pos → 전부 False"] = (not out.any()) and (not out2.any())

    # [3] 초기 방향 하드코딩 금지 (V3-D)
    multi = [i for i, s in enumerate(pool) if len(s["dirs"]) > 1]
    if multi:
        seen = set()
        for _ in range(60):
            env.reset(options={"index": multi[0]}); seen.add(env.dir)
        results["[3] 초기방향 집합 사용"] = len(seen) > 1
    else:
        results["[3] 초기방향 집합 사용"] = True   # 후보가 1개뿐이면 검증 불가

    # [4] 시나리오 생성기: k 제약 하 도달성 (V1)
    results["[4] k 제약 도달성 검증됨"] = all(
        d in s["astar"] and s["astar"][d] is not None for s in pool for d in s["dirs"])

    # [5] J 정규화 상수 (G1-Q2)
    expect = math.sqrt(cfg.NX**2 + cfg.NY**2 + cfg.NZ**2) * cfg.CELL_M * SPEC.kg_per_m
    results["[5] DIAG_J 정규화"] = abs(DIAG_J - expect) < 1e-9 and abs(SCALE * DIAG_J - 1.0) < 1e-12

    # [6~9] 랜덤 마스킹 롤아웃 불변식
    rev_ok = turn_ok = pbs_ok = inside_ok = True
    invalid_seen = 0
    for ep in range(n_ep):
        obs, _ = env.reset()
        for _ in range(cfg.max_steps):
            m = env.action_masks()
            if m[OPP26[env.dir]]:
                rev_ok = False                                   # [6] 역방향 금지
            if env.k > 0 and env.s < env.k and m[:26].sum() > 0:
                allowed = np.where(m[:26])[0]
                if any(int(a) != env.dir for a in allowed):
                    turn_ok = False                              # [7] s<k 면 턴 불가
            a = int(rng.choice(np.where(m)[0]))
            prev_pos, prev_dir, prev_g = env.pos, env.dir, env.g_J
            phi_prev = Phi(prev_pos, env.goal, env.pad)
            obs, r, term, trunc, info = env.step(a)
            if a != STAY:
                dJ = env.g_J - prev_g
                if cfg.stage in ("1a", "1b"):                    # 커리큘럼 보상식
                    exp_r = -cfg.reach_step_cost + (cfg.reach_goal_bonus if term else 0.0)
                else:
                    exp_r = -dJ / J_MAX + cfg.gamma * Phi(env.pos, env.goal, env.pad) - phi_prev
                if abs(exp_r - r) > 1e-9:
                    pbs_ok = False                               # [8] 보상식 일치
            if not (0 <= env.pos[0] < cfg.NX and 0 <= env.pos[1] < cfg.NY and 0 <= env.pos[2] < cfg.NZ):
                inside_ok = False                                # [9] 격자 이탈 없음
            invalid_seen += int(info["n_invalid"] > 0)
            if term or trunc:
                break
    results["[6] 역방향 상시 마스킹"] = rev_ok
    results["[7] s<k 면 턴 마스킹"] = turn_ok if cfg.k_override != 0 else True
    results[f"[8] 보상식 일치 (stage={cfg.stage})"] = pbs_ok
    results["[9] 격자 이탈 없음"] = inside_ok
    results["[10] Φ(종단)=0"] = (Phi((4, 4, 4), (4, 4, 4)) == 0.0)

    # [11~12] 관종 (Step 3~, §36-6) — 중력관이 풀에 있을 때만
    grav = [i for i, s in enumerate(pool) if is_grav(s)]
    if grav:
        up_ok, chan_ok, fair_ok = True, True, True
        genv = PipeRoutingEnvStep1(pool, eval_mode=False, seed=seed + 1)
        for j in range(n_ep):
            gi = grav[j % len(grav)]
            genv.reset(options={"index": gi})
            for _ in range(cfg.max_steps):
                m = genv.action_masks()
                if m[:26][UPHILL].any():
                    up_ok = False                                # [11] 중력관 오르막 금지
                if m[:26].any() and not np.array_equal(
                        m, action_mask_27(genv.pad, genv.pos, genv.dir, genv.s, genv.goal, genv.k, gravity=True)):
                    fair_ok = False                              # [11] 환경 = A* 마스크 (§24-1)
                o = genv._obs()
                if o[80] != 1.0 or o[81] != 0.0:
                    chan_ok = False                              # [12] 관종 채널
                obs, r, term, trunc, info = genv.step(int(rng.choice(np.where(m)[0])))
                if term or trunc:
                    break
        # 중력관 A* 기준선이 오르막을 쓰지 않는가 · 목표가 시작보다 높지 않은가
        for gi in grav:
            s = pool[gi]
            if s["goal"][2] > s["start"][2]:
                up_ok = False
            for sol in s["astar"].values():
                zs = [st_[0][2] for st_ in sol["states"]]
                if any(b_ > a_ for a_, b_ in zip(zs[:-1], zs[1:])):
                    up_ok = False
        penv = PipeRoutingEnvStep1(pool, eval_mode=True)
        pres = [i for i, s in enumerate(pool) if not is_grav(s)]
        if pres:
            penv.reset(options={"index": pres[0]})
            chan_ok &= bool(penv._obs()[80] == 0.0)
        results["[11] 중력관 역구배 금지 (G0) · 환경=A* 마스크"] = up_ok and fair_ok
        results["[12] 관종 채널 obs[80]=is_gravity · obs[81]=0"] = chan_ok
    return results

_res = run_checklist_tests()
if any(map(is_grav, TRAIN_POOL)) and not any(map(is_grav, EVAL_POOL)):
    print("  (평가 풀에 중력관이 없다 — [11~12] 는 학습 풀로 검사)")
    _res.update({k_: v for k_, v in run_checklist_tests(TRAIN_POOL).items() if k_.startswith("[1") and k_[2] in "12"})
for k_, v in _res.items():
    print(("  PASS  " if v else "  FAIL  ") + k_)
assert all(_res.values()), "§11 체크리스트 실패 — 학습 금지"
print("\n§11 체크리스트 전부 통과")

## 셀 6. A\* 기준선 저장

평가 시나리오 48개의 k 제약 A\* 최적해를 `step1_baseline_astar.json` 으로 남긴다.
`length_ratio` / `j_ratio` 는 전부 이 기준선 대비 값이다.

In [ ]:
#@title 6. A* 기준선 → step1_baseline_astar.json
BASELINE_PATH = "step1_baseline_astar.json"
_base = []
for i, sc in enumerate(EVAL_POOL):
    d = sc["dirs"][0]
    s = sc["astar"][d]
    _base.append(dict(index=i, start=list(sc["start"]), goal=list(sc["goal"]), start_dir=int(d),
                      J_kg=round(float(s["J"]), 4), len_m=round(float(s["len_m"]), 4),
                      n_cells=int(s["n_cells"]), bends90=int(s["bends90"]), bends45=int(s["bends45"]),
                      astar_expanded=int(s["expanded"])))
json.dump(dict(
    grid=[cfg.NX, cfg.NY, cfg.NZ], cell_m=cfg.CELL_M, nominal=cfg.nominal, k=K,
    note="k 제약 하 A* 최적해. 모든 length_ratio/j_ratio 의 분모.",
    mean=dict(J_kg=round(float(np.mean([b["J_kg"] for b in _base])), 3),
              len_m=round(float(np.mean([b["len_m"] for b in _base])), 3),
              n_cells=round(float(np.mean([b["n_cells"] for b in _base])), 2),
              elbows=round(float(np.mean([b["bends90"] + b["bends45"] for b in _base])), 2)),
    scenarios=_base), open(BASELINE_PATH, "w"), ensure_ascii=False, indent=1)
print(f"{BASELINE_PATH} 저장 | 평가 {len(_base)}개 평균: J {np.mean([b['J_kg'] for b in _base]):.2f} kg | "
      f"길이 {np.mean([b['len_m'] for b in _base]):.2f} m | 칸수 {np.mean([b['n_cells'] for b in _base]):.1f} | "
      f"엘보 {np.mean([b['bends90']+b['bends45'] for b in _base]):.2f}")

## 셀 7. h_net 사전학습 (A\* 라벨, 3,000 배치)

라벨은 **학습 풀** A\* 최적해의 접미 비용(= 그 상태에서 남은 최적 J). 평가셋은 쓰지 않는다.
A\* 해를 보상으로 쓰는 것이 아니라 **h_net 초기값만** 잡아준다 (CLAUDE.md §1 철학과 충돌하지 않는다).

> **배치 수를 500 → 3,000 으로 올렸다 (이 격자에서 실측).** 라벨 상태가 31k 개라 500 배치는 4 에폭뿐이고,
> 그 h 는 맨해튼보다도 못하다: val MAE **500배치 5.19kg / 1,500 2.10 / 3,000 1.51 / 6,000 1.14**
> (맨해튼 2.88 · octile3d 3.50). 3,000 배치 비용은 **3초**고, 졸업 기준 h MAE ≤ 2.0kg 아래로 내려온다.
> G3 의 500 은 20×20×10 기준값이었다.

> 참고: G3 §16-9 에서 **기존 구조**의 사전학습은 성능 기여가 없었다(Δ=+0.009, 노이즈).
> 진짜 GNARL 에서는 h 가 행동을 직접 고르므로 기여 여부가 다를 수 있다 — **미결**. 셀 10 §17-4 로 감시한다.

In [ ]:
#@title 7. h_net 사전학습
import torch
import torch.nn as nn
import torch.nn.functional as F
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor

class GNARLFeaturesExtractor(BaseFeaturesExtractor):
    def __init__(self, observation_space, features_dim: int = 64, hidden: int = 128,
                 clip_mode: str = "manhattan"):
        super().__init__(observation_space, features_dim)
        n_in = int(observation_space.shape[0])
        self.clip_mode = clip_mode
        self.feat_net = nn.Sequential(                       # 일반 특징 (features_dim - 1)
            nn.Linear(n_in, hidden), nn.ReLU(),
            nn.Linear(hidden, features_dim - 1), nn.ReLU(),
        )
        self.h_net = nn.Sequential(                          # h(n) 추정 (스칼라 1개)
            nn.Linear(n_in, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, 1),
        )
        self.register_buffer("dims_t", torch.tensor(DIMS, dtype=torch.float32))
        # 1칸당 J 를 J_MAX 로 정규화한 상수 (§6: 정규화 없으면 h 학습 실패)
        self.register_buffer("unit_t", torch.tensor(cfg.CELL_M * SPEC.kg_per_m / J_MAX,
                                                    dtype=torch.float32))

    def _delta_cells(self, obs: torch.Tensor) -> torch.Tensor:
        return (obs[:, 0:3] - obs[:, 3:6]).abs() * self.dims_t        # (B,3) 칸 단위

    def h_bound(self, obs: torch.Tensor) -> torch.Tensor:
        d = self._delta_cells(obs)
        if self.clip_mode == "octile3d":
            srt, _ = torch.sort(d, dim=-1, descending=True)
            a, b, c = srt[:, 0:1], srt[:, 1:2], srt[:, 2:3]
            cells = (a - b) + (b - c) * SQRT2 + c * SQRT3
        else:                                                          # "manhattan" (§6 원문)
            cells = d.sum(dim=-1, keepdim=True)
        return cells * self.unit_t

    def h_raw(self, obs: torch.Tensor) -> torch.Tensor:
        return F.softplus(self.h_net(obs))                             # h >= 0

    def h(self, obs: torch.Tensor) -> torch.Tensor:
        raw = self.h_raw(obs)
        if self.clip_mode == "none":
            return raw
        return torch.minimum(raw, self.h_bound(obs))                   # §6 안전 클리핑

    def forward(self, obs: torch.Tensor) -> torch.Tensor:
        return torch.cat([self.feat_net(obs), self.h(obs)], dim=-1)

# h_net 은 언제나 '상태 1개의 obs(OBS_DIM 차원)' 를 입력으로 받는다.
# gnarl_mode 에서는 환경 관측이 이웃까지 실어 2155차원이 되므로, 아래 공간을 따로 둔다.
OWN_SPACE = spaces.Box(-1.0, 1.0, (OBS_DIM,), dtype=np.float32)

# --- §11 체크리스트 [h(n) admissibility 클리핑] 검증 ---
_ext = GNARLFeaturesExtractor(OWN_SPACE, cfg.features_dim, cfg.hidden, cfg.h_clip_mode)
_obs_batch = torch.as_tensor(np.stack([
    build_obs(sc["pad"], sc["start"], sc["dirs"][0], K, 0.0, sc["goal"], gravity=is_grav(sc)) for sc in EVAL_POOL]))
with torch.no_grad():
    _h, _b, _f = _ext.h(_obs_batch), _ext.h_bound(_obs_batch), _ext(_obs_batch)
# clip_mode="none" 은 상한을 걸지 않는 설정이므로 이 검사 대상이 아니다.
if cfg.h_clip_mode == "none":
    print(f"클리핑 모드=none | 상한 미적용 (h>bound 비율 {float((_h > _b).float().mean()):.0%}) "
          f"| features={_f.shape}")
else:
    assert bool((_h <= _b + 1e-6).all()), "admissibility 클리핑 실패"
    print(f"클리핑 모드={cfg.h_clip_mode} | h<=bound 전부 통과 | features={_f.shape} "
          f"| bound 평균 {_b.mean():.3f} (정규화 J)")
assert _f.shape[1] == cfg.features_dim

def build_h_dataset(pool):
    X, Y = [], []
    for sc in pool:
        for d, sol in sc["astar"].items():
            Jt = sol["J"]
            for (pos, dd, ss, gg) in sol["states"]:
                X.append(build_obs(sc["pad"], pos, dd, ss, gg, sc["goal"], gravity=is_grav(sc)))
                Y.append((Jt - gg) / J_MAX)          # 남은 최적 J (정규화)
    return np.stack(X), np.asarray(Y, dtype=np.float32)[:, None]

PRETRAIN_INFO = {}    # 실행 기록용 (배치 수, MAE) — 사전학습이 실제로 무엇을 했는지 남긴다

def pretrain_h_net(extractor, X, Y, epochs=30, bs=256, lr=1e-3, val_frac=0.15, verbose=True):
    n = len(X); idx = np.random.default_rng(SEED).permutation(n)
    nv = int(n * val_frac)
    vi, ti = idx[:nv], idx[nv:]
    Xt = torch.as_tensor(X[ti]); Yt = torch.as_tensor(Y[ti])
    Xv = torch.as_tensor(X[vi]); Yv = torch.as_tensor(Y[vi])
    opt = torch.optim.Adam(extractor.h_net.parameters(), lr=lr)
    hist = []
    n_batches = 0
    for ep in range(epochs):
        perm = torch.randperm(len(Xt))
        for i in range(0, len(Xt), bs):
            b = perm[i:i+bs]
            loss = F.mse_loss(extractor.h_raw(Xt[b]), Yt[b])
            opt.zero_grad(); loss.backward(); opt.step()
            n_batches += 1
        with torch.no_grad():
            vr = extractor.h_raw(Xv)
            vmae = float((vr - Yv).abs().mean())
            clipped = float((vr > extractor.h_bound(Xv)).float().mean())
        hist.append((ep, vmae, clipped))
        if verbose and (ep % 5 == 0 or ep == epochs - 1):
            print(f"  ep {ep:3d} | val MAE {vmae:.4f} (= {vmae*J_MAX:.2f} kg) | 클리핑 발생률 {clipped:.1%}")
    PRETRAIN_INFO.update(n_batches=n_batches, n_states=int(n), val_mae_kg=float(hist[-1][1]*J_MAX),
                         clip_rate=float(hist[-1][2]))
    if verbose:
        print(f"  → 총 {n_batches} 배치 / {n} 상태 (라벨 출처: TRAIN_POOL 의 A* 최적해 접미 비용. 평가셋 미사용)")
    return hist


def pretrain_h_batches(extractor, X, Y, n_batches=500, bs=256, lr=1e-3, val_frac=0.15):
    """정확히 n_batches 만큼만 돈다 (에폭이 아니라 배치로 센다)."""
    n = len(X); idx = np.random.default_rng(SEED).permutation(n); nv = int(n * val_frac)
    Xt, Yt = torch.as_tensor(X[idx[nv:]]), torch.as_tensor(Y[idx[nv:]])
    Xv, Yv = torch.as_tensor(X[idx[:nv]]), torch.as_tensor(Y[idx[:nv]])
    opt = torch.optim.Adam(extractor.h_net.parameters(), lr=lr)
    rng = np.random.default_rng(SEED)
    hist = []
    for b in range(n_batches):
        sel = torch.as_tensor(rng.integers(0, len(Xt), size=min(bs, len(Xt))))
        loss = F.mse_loss(extractor.h_raw(Xt[sel]), Yt[sel])
        opt.zero_grad(); loss.backward(); opt.step()
        if (b + 1) % 100 == 0 or b == n_batches - 1:
            with torch.no_grad():
                mae = float((extractor.h_raw(Xv) - Yv).abs().mean())
            hist.append((b + 1, mae * J_MAX))
            print(f"  배치 {b+1:4d} | val MAE {mae*J_MAX:.3f} kg")
    return hist

Xh, Yh = build_h_dataset(TRAIN_POOL)
print(f"h(n) 라벨: {len(Xh)} 상태 (학습 풀 A* 최적해의 접미 비용) | "
      f"평균 남은 J {Yh.mean()*J_MAX:.2f} kg")
H_EXTRACTOR_SD, PRETRAIN_HIST = None, []
if cfg.pretrain_h:
    _pre = GNARLFeaturesExtractor(OWN_SPACE, cfg.features_dim, cfg.hidden, cfg.h_clip_mode)
    PRETRAIN_HIST = pretrain_h_batches(_pre, Xh, Yh, n_batches=cfg.pretrain_batches)
    H_EXTRACTOR_SD = {k_: v.clone() for k_, v in _pre.h_net.state_dict().items()}
    with torch.no_grad():
        Xt_, Yt_ = torch.as_tensor(Xh), torch.as_tensor(Yh)
        mae_net = float((_pre.h_raw(Xt_) - Yt_).abs().mean()) * J_MAX
        _pre.clip_mode = "manhattan"; mae_man = float((_pre.h_bound(Xt_) - Yt_).abs().mean()) * J_MAX
        _pre.clip_mode = "octile3d";  mae_oct = float((_pre.h_bound(Xt_) - Yt_).abs().mean()) * J_MAX
        _pre.clip_mode = cfg.h_clip_mode
    PRETRAIN_INFO.update(n_batches=cfg.pretrain_batches, n_states=int(len(Xh)),
                         mae_net_kg=mae_net, mae_manhattan_kg=mae_man, mae_octile3d_kg=mae_oct)
    print(f"\n사전학습 완료 — MAE(kg): 학습 h {mae_net:.2f} | 맨해튼 {mae_man:.2f} | octile3d {mae_oct:.2f}")
    print(f"  → 졸업 기준 h MAE ≤ {cfg.grad_h_mae_kg} kg "
          f"{'달성' if mae_net <= cfg.grad_h_mae_kg else '미달 (학습 중 개선 여부를 §17-2 로 감시)'}")
else:
    print("사전학습 건너뜀 (cfg.pretrain_h=False)")

## 셀 7b. 순위 라벨 + h 순위 정칙항 (CLAUDE.md §21)

붕괴의 실체는 **h 의 순위 손상 하나**다 (§20-4: Spearman ρ 0.93 → -0.06, success 가 정확히 따라간다).
PPO 는 h 를 '남은 최적 J' 로 되돌리는 항이 없어 단조로 파괴한다.

- **순위 대상은 h 가 아니라 `q = ΔJ + h`** — 정책이 비교하는 `f = g + ΔJ + h` 에서 g 가 상쇄된 것 (§21-1)
- **마진은 τ 단위** — τ 보다 큰 차이는 이미 결정된 것이고 τ 이하만 순위를 바꾼다 (§21-2)
- **강도는 λ 가 아니라 `h_rank_batches`(롤아웃당 스텝 수)** — 별도 Adam 이라 손실에 λ 를 곱해도 거의 상쇄된다
- **PPO 본체 무수정** — 콜백의 별도 옵티마이저를 `_on_rollout_start` 에서만 돌린다 (§21-3)
- 라벨은 **학습 풀에서만**. 평가 풀 라벨은 계기판 전용이다 (§21-4)


In [ ]:
#@title 7b. 순위 라벨 + h 순위 정칙항 (CLAUDE.md §21)
# PPO 본체는 건드리지 않는다 (§0 변경 불가, §6 구성 5).
# h_core 전용 Adam 을 콜백의 _on_rollout_start 에서만 돌린다 = train() 직후 · 다음 롤아웃 수집 직전.
from stable_baselines3.common.callbacks import BaseCallback

RANK_CACHE = (CACHE.replace("g3_scen_", "step1_rank_").replace(".pkl", "")
              + f"_o{OBS_DIM}{cfg.obs_align_norm}"          # obs 가 바뀌면 캐시도 갈린다 (§22-4)
              + ("" if cfg.legacy_dj_edge_bug else "_djfix")  # S1-B1: 버그 ΔJ 라벨과 섞지 않는다 (§27)
              + ("_off" if cfg.h_rank_offpath else "")        # (b) 경로 밖 라벨 포함 (§28)
              + ".npz")
if cfg.h_rank_offpath and not os.path.exists(RANK_CACHE):
    raise FileNotFoundError(f"{RANK_CACHE} 없음 — results/offpath_labels.py collect → label 로 먼저 만든다 (§28-1)")

def _rank_rows_for_state(sc, pos, dd, ss, gg):
    """결정 시점 하나의 (이웃 obs, ΔJ, t). t = ΔJ + y, y = 그 이웃에서의 A* 남은 최적 J (§21-1).

    정책이 비교하는 것은 f = g + ΔJ + h(s') 이고 g 는 이웃 공통이라 상쇄된다.
    따라서 순위 대상은 q = ΔJ + h(s') 이고 정답은 t = ΔJ + y 다.
    h 만 y 에 맞추면 엘보 비용이 순위에서 빠진다 (§6: '행동을 가르는 것은 ΔJ + h').
    """
    m = action_mask_27(sc["pad"], pos, dd, ss, sc["goal"], K, gravity=is_grav(sc))
    cand = [int(i) for i in np.where(m[:26])[0]]
    if len(cand) < 2:
        return None
    dJ_all = move_J_all(dd)
    rows = []
    for i in cand:
        d = DIRS26[i]
        npos = (pos[0] + d[0], pos[1] + d[1], pos[2] + d[2])
        nss = min(ss + 1, K) if i == dd else 1
        dj = float(dJ_all[i]) / J_MAX
        if npos == tuple(sc["goal"]):
            y = 0.0
        else:
            r = astar_gnarl(sc["pad"], npos, sc["goal"], i, s0=nss, gravity=is_grav(sc))
            if r is None:
                return None                     # 도달 불가 이웃이 있으면 이 시점은 버린다
            y = float(r["J"]) / J_MAX
        rows.append((build_obs(sc["pad"], npos, i, nss, gg + float(dJ_all[i]), sc["goal"], gravity=is_grav(sc)),
                     dj, dj + y))
    return rows


def build_rank_dataset(pool, desc="rank labels"):
    """A* 경로 위 **결정 시점**(유효 이웃 ≥ 2)마다 모든 유효 이웃을 라벨링한다 (§21-4).

    경로 밖 이웃의 남은 J 는 그 이웃에서 A* 를 한 번 더 돌려야 나온다 (실측 0.025초/이웃).
    라벨은 **학습 풀에서만** 만든다 — 평가 풀 라벨은 계기판 전용이다 (§21-4, §0).
    """
    obs, dj, t, ptr = [], [], [], [0]
    for sc in tqdm(pool, desc=desc):
        goal = tuple(sc["goal"])
        for d, sol in sc["astar"].items():
            for (pos, dd, ss, gg) in sol["states"]:
                if tuple(pos) == goal:
                    continue
                rows = _rank_rows_for_state(sc, tuple(pos), int(dd), int(ss), float(gg))
                if not rows:
                    continue
                for (o, a, b) in rows:
                    obs.append(o); dj.append(a); t.append(b)
                ptr.append(len(obs))
    obs = np.stack(obs).astype(np.float32)
    dj = np.asarray(dj, dtype=np.float32); t = np.asarray(t, dtype=np.float32)
    ptr = np.asarray(ptr, dtype=np.int64)
    pi, pj = [], []                              # t_i < t_j 인 모든 쌍 (전역 행 인덱스)
    for a, b in zip(ptr[:-1], ptr[1:]):
        for i in range(a, b):
            for j in range(a, b):
                if t[i] < t[j] - 1e-9:
                    pi.append(i); pj.append(j)
    return dict(obs=obs, dj=dj, t=t, ptr=ptr,
                pi=np.asarray(pi, dtype=np.int64), pj=np.asarray(pj, dtype=np.int64))


def _load_or_build_rank():
    if os.path.exists(RANK_CACHE):
        z = np.load(RANK_CACHE)
        d = {k: z[k] for k in ("obs", "dj", "t", "ptr", "pi", "pj")}
        e = {k: z["e_" + k] for k in ("obs", "dj", "t", "ptr", "pi", "pj")}
        print(f"순위 라벨 캐시 로드: {RANK_CACHE}")
    else:
        t0 = time.time()
        d = build_rank_dataset(TRAIN_POOL, "rank labels (train)")
        e = build_rank_dataset(EVAL_POOL, "rank labels (eval, 계기판 전용)")
        np.savez_compressed(RANK_CACHE, **d, **{("e_" + k): v for k, v in e.items()})
        print(f"순위 라벨 생성 {time.time()-t0:.0f}s → {RANK_CACHE}")
    print(f"  학습 라벨: 결정시점 {len(d['ptr'])-1:,} · 이웃 {len(d['t']):,} · 쌍 {len(d['pi']):,}")
    print(f"  계기판    : 결정시점 {len(e['ptr'])-1:,} · 이웃 {len(e['t']):,} (평가 풀 — 학습에 쓰지 않는다)")
    return d, e


def _subset_rank_by_scen(d, pool, n):
    """§41: 학습 풀 앞 n 개 시나리오의 라벨 그룹만 남긴다. 그룹의 시나리오 = 그룹 첫 행의 목표 좌표 (obs[3:6]).
    목표가 앞 n 개와 뒤쪽 시나리오에 겹치는 그룹은 앞쪽으로 센다 (학습 풀 목표 중복은 4쌍뿐, §32-4)."""
    allowed = {tuple(int(v) for v in sc["goal"]) for sc in pool[:n]}
    ptr = d["ptr"]
    goals = np.rint(d["obs"][ptr[:-1], 3:6] * DIMS).astype(np.int64)
    keep = np.array([tuple(g) in allowed for g in goals])
    lens = np.diff(ptr)
    row_keep = np.repeat(keep, lens)
    new_idx = np.cumsum(row_keep) - 1
    pm = row_keep[d["pi"]] & row_keep[d["pj"]]
    out = dict(obs=d["obs"][row_keep], dj=d["dj"][row_keep], t=d["t"][row_keep],
               ptr=np.concatenate([[0], np.cumsum(lens[keep])]).astype(np.int64),
               pi=new_idx[d["pi"][pm]].astype(np.int64), pj=new_idx[d["pj"][pm]].astype(np.int64))
    print(f"  라벨량 제한 (§41): 학습 풀 앞 {n} 시나리오 → 결정시점 {int(keep.sum()):,}/{len(keep):,} · "
          f"이웃 {int(row_keep.sum()):,} · 쌍 {len(out['pi']):,}")
    return out


RANK_TRAIN, RANK_EVAL = (None, None)
if cfg.h_rank_batches > 0 or cfg.h_rank_metrics:
    RANK_TRAIN, RANK_EVAL = _load_or_build_rank()
    if cfg.h_rank_n_scen is not None:
        RANK_TRAIN = _subset_rank_by_scen(RANK_TRAIN, TRAIN_POOL, int(cfg.h_rank_n_scen))
    _RT = {k: torch.as_tensor(v) for k, v in RANK_TRAIN.items()}
    _RE = {k: torch.as_tensor(v) for k, v in RANK_EVAL.items()}


def _cur_tau(policy) -> float:
    tau = float(policy.log_tau.detach().exp()) if cfg.gnarl_mode else 1.0
    return max(tau, cfg.gnarl_tau_min) if cfg.gnarl_tau_min > 0 else tau


def h_rank_loss(policy, rng) -> "torch.Tensor":
    """쌍별 로지스틱(RankNet). 마진은 τ 단위 — τ 보다 큰 차이는 이미 결정된 것이다 (§21-2)."""
    n_pairs = len(_RT["pi"])
    sel = torch.as_tensor(rng.integers(0, n_pairs, size=min(cfg.h_rank_bs, n_pairs)))
    i, j = _RT["pi"][sel], _RT["pj"][sel]
    rows = torch.cat([i, j])
    h = F.softplus(policy.h_core(_RT["obs"][rows].to(policy.device))).squeeze(-1).cpu()
    q = _RT["dj"][rows] + h
    n = len(i)
    l = F.softplus((q[:n] - q[n:]) / _cur_tau(policy))
    if cfg.h_rank_weight == "cost":                 # C2 (§30): 틀리면 비싼 쌍을 더 크게 배운다
        w = _RT["t"][j] - _RT["t"][i]               # > 0 (쌍은 t_i < t_j 로만 만든다)
        return (l * w).sum() / w.sum()
    return l.mean()


@torch.no_grad()
def h_rank_metrics(policy) -> dict:
    """§21-5 계기판. 평가 풀 라벨로만 잰다 (학습에는 쓰지 않는다)."""
    if RANK_EVAL is None:
        return {}
    h = F.softplus(policy.h_core(_RE["obs"].to(policy.device))).squeeze(-1).cpu()
    q = _RE["dj"] + h
    ptr = RANK_EVAL["ptr"]; t = RANK_EVAL["t"]
    top1, spread = [], []
    for a, b in zip(ptr[:-1], ptr[1:]):
        qq = q[a:b].numpy()
        top1.append(float(int(qq.argmin()) == int(t[a:b].argmin())))
        spread.append(float(h[a:b].numpy().std()))
    return dict(rank_top1=float(np.mean(top1)),
                h_spread=float(np.mean(spread)),
                rank_spearman=h_spearman_eval(policy))


def _rankdata(v):
    o = np.argsort(v, kind="mergesort"); r = np.empty(len(v), float); r[o] = np.arange(len(v))
    vs = v[o]; i = 0
    while i < len(vs):
        j = i
        while j + 1 < len(vs) and vs[j + 1] == vs[i]:
            j += 1
        if j > i:
            r[o[i:j + 1]] = (i + j) / 2.0
        i = j + 1
    return r


@torch.no_grad()
def h_spearman_eval(policy, max_traj=48) -> float:
    """A* 궤적 위 h vs 실제 남은 J 의 Spearman ρ (§20-4 와 같은 양, 연속성용)."""
    rhos = []
    for sc in EVAL_POOL[:max_traj]:
        for d, sol in sc["astar"].items():
            st = sol["states"]
            if len(st) < 4:
                continue
            X = np.stack([build_obs(sc["pad"], p, dd, ss, gg, sc["goal"], gravity=is_grav(sc)) for (p, dd, ss, gg) in st])
            Y = np.array([(sol["J"] - gg) / J_MAX for (_, _, _, gg) in st], dtype=np.float32)
            hh = F.softplus(policy.h_core(torch.as_tensor(X).to(policy.device))).squeeze(-1).cpu().numpy()
            ra, rb = _rankdata(hh), _rankdata(Y)
            if ra.std() > 0 and rb.std() > 0:
                rhos.append(float(np.corrcoef(ra, rb)[0, 1]))
    return float(np.mean(rhos)) if rhos else float("nan")


class HRankCallback(BaseCallback):
    """§21-3. PPO 손실을 건드리지 않고 h_core 만 별도 Adam 으로 되돌린다.

    갱신 시점은 _on_rollout_start 하나뿐이다:
      _on_step(수집 중)      → 한 롤아웃 버퍼가 여러 정책의 혼합이 된다
      _on_rollout_end(train 전) → ratio != 1 에서 PPO 업데이트가 시작돼 클리핑 가정이 깨진다
      _on_rollout_start      → 갱신된 h 가 '다음 롤아웃의 행동 정책' 이 되어 on-policy 가 유지된다
    """

    def __init__(self, verbose=0):
        super().__init__(verbose)
        self.opt = None
        self.rng = np.random.default_rng(SEED)
        self.last_loss = float("nan")

    def _on_training_start(self) -> None:
        if cfg.h_rank_batches <= 0:
            return
        ps = [p for p in self.model.policy.h_core.parameters() if p.requires_grad]
        if not ps:
            print("h_rank: h_core 가 동결돼 있어 순위 정칙항을 건너뛴다")
            return
        self.opt = torch.optim.Adam(ps, lr=cfg.h_rank_lr)
        print(f"h_rank: 롤아웃당 {cfg.h_rank_batches} 스텝 · lr {cfg.h_rank_lr} · 쌍 {cfg.h_rank_bs}/배치")

    def _on_rollout_start(self) -> None:
        if self.opt is None:
            return
        for _ in range(cfg.h_rank_batches):
            loss = h_rank_loss(self.model.policy, self.rng)
            self.opt.zero_grad(); loss.backward(); self.opt.step()
        self.last_loss = float(loss.detach())

    def _on_step(self) -> bool:
        return True


## 셀 8. 모델 생성 — 진짜 GNARL

```
obs → 유효 이웃 27개 → h_net(배치) → f = g + ΔJ + h → softmin(-f/τ) → 행동
```
PPO 본체(rollout / advantage / clip loss)는 sb3-contrib 원본 그대로다 (CLAUDE.md §0 변경불가).
**`cfg.resume_from` 이 있으면 그 체크포인트에서 이어서 학습한다.**

In [ ]:
#@title 8. TrueGNARLPolicy + 모델
from sb3_contrib.common.maskable.policies import MaskableActorCriticPolicy
from stable_baselines3.common.torch_layers import FlattenExtractor

class TrueGNARLPolicy(MaskableActorCriticPolicy):
    """행동 logits 를 f(n)=g+ΔJ+h 에서 직접 만든다. PPO 알고리즘 본체는 건드리지 않는다."""

    def __init__(self, observation_space, action_space, lr_schedule, *args, **kwargs):
        kwargs["features_extractor_class"] = FlattenExtractor   # 2155차원을 MLP 에 태우지 않는다
        kwargs.pop("features_extractor_kwargs", None)
        super().__init__(observation_space, action_space, lr_schedule, *args, **kwargs)
        hid = cfg.hidden
        self.h_core = nn.Sequential(                      # h(n): 상태 1개 → 스칼라
            nn.Linear(OBS_DIM, hid), nn.ReLU(),
            nn.Linear(hid, hid), nn.ReLU(),
            nn.Linear(hid, 1),
        )
        self.v_core = nn.Sequential(                      # value: 자기 obs 만 본다
            nn.Linear(OBS_DIM, hid), nn.ReLU(),
            nn.Linear(hid, hid), nn.ReLU(),
            nn.Linear(hid, 1),
        )
        self.log_tau = nn.Parameter(torch.tensor(float(np.log(cfg.gnarl_tau_init))),
                                    requires_grad=bool(cfg.gnarl_learn_tau))
        self.register_buffer("dims_t", torch.tensor(DIMS, dtype=torch.float32))
        self.register_buffer("unit_t", torch.tensor(cfg.CELL_M * SPEC.kg_per_m / J_MAX,
                                                    dtype=torch.float32))
        # super().__init__ 이 만든 optimizer 에는 위 모듈들이 없다 → 다시 만든다
        self.optimizer = self.optimizer_class(self.parameters(), lr=lr_schedule(1),
                                              **self.optimizer_kwargs)

    # ---------- f = g + ΔJ + h ----------
    def _split(self, obs):
        B = obs.shape[0]
        own = obs[:, :OBS_DIM]
        nbr = obs[:, OBS_DIM:OBS_DIM + N_ACTIONS * OBS_DIM].reshape(B, N_ACTIONS, OBS_DIM)
        dJ = obs[:, -N_ACTIONS:]
        return own, nbr, dJ

    def h_bound(self, x):                                  # 맨해튼 상한 (§6)
        d = (x[:, 0:3] - x[:, 3:6]).abs() * self.dims_t
        return d.sum(dim=-1, keepdim=True) * self.unit_t

    def h_admissible(self, x):
        raw = F.softplus(self.h_core(x))
        if cfg.h_clip_mode == "none":
            return raw
        return torch.minimum(raw, self.h_bound(x))

    def f_values(self, obs):
        own, nbr, dJ = self._split(obs)
        B = obs.shape[0]
        h = self.h_admissible(nbr.reshape(B * N_ACTIONS, OBS_DIM)).reshape(B, N_ACTIONS)
        g = own[:, 33:34]                                  # 누적 J (정규화) — 이웃 공통, softmax 에서 상쇄
        return g + dJ + h                                  # (B,27)

    def _logits(self, obs):
        # τ 하한: 1차 런에서 τ 가 0.05 → 0.0117 까지 내려가며 정책이 틀린 f 순위에
        # 확신을 키웠다 (§19-2). cfg.gnarl_tau_min 을 주면 그 아래로 못 내려간다.
        tau = self.log_tau.exp().clamp(min=max(1e-4, float(cfg.gnarl_tau_min)))
        return -self.f_values(obs) / tau

    # ---------- sb3 API ----------
    def _dist(self, obs, action_masks=None):
        d = self.action_dist.proba_distribution(action_logits=self._logits(obs))
        if action_masks is not None:
            d.apply_masking(action_masks)
        return d

    def get_distribution(self, obs, action_masks=None):
        return self._dist(obs, action_masks)

    def predict_values(self, obs):
        return self.v_core(obs[:, :OBS_DIM])

    def forward(self, obs, deterministic=False, action_masks=None):
        d = self._dist(obs, action_masks)
        actions = d.get_actions(deterministic=deterministic)
        return actions, self.predict_values(obs), d.log_prob(actions)

    def _predict(self, observation, deterministic=False, action_masks=None):
        return self._dist(observation, action_masks).get_actions(deterministic=deterministic)

    def evaluate_actions(self, obs, actions, action_masks=None):
        d = self._dist(obs, action_masks)
        return self.predict_values(obs), d.log_prob(actions), d.entropy()

# --- τ 초기값 근거: 이웃 간 f 차이의 실제 크기를 잰다 ---
if cfg.gnarl_mode:
    _genv = PipeRoutingEnvStep1(EVAL_POOL, eval_mode=True)
    _spread = []
    for i in range(min(12, len(EVAL_POOL))):
        o, _ = _genv.reset(options={"index": i})
        m = _genv.action_masks()
        own, nbr, dJ = (o[:OBS_DIM], o[OBS_DIM:OBS_DIM+N_ACTIONS*OBS_DIM].reshape(N_ACTIONS, OBS_DIM),
                        o[-N_ACTIONS:])
        hman = np.abs(nbr[:, 0:3] - nbr[:, 3:6]) * DIMS
        hman = hman.sum(-1) * (cfg.CELL_M * SPEC.kg_per_m / J_MAX)
        f = dJ + hman                                       # h 를 맨해튼으로 대체한 근사
        _spread.append(float(np.std(f[m])))
    print(f"이웃 간 f 차이의 표준편차 ≈ {np.mean(_spread):.4f} (정규화 J 단위) "
          f"→ τ={cfg.gnarl_tau_init} 이면 logit 폭 ≈ {np.mean(_spread)/cfg.gnarl_tau_init:.2f}")
    print(f"   (τ=1.0 이면 {np.mean(_spread):.3f} — 사실상 균등분포)")


from sb3_contrib import MaskablePPO
from sb3_contrib.common.wrappers import ActionMasker
from sb3_contrib.common.maskable.utils import is_masking_supported
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.utils import set_random_seed
set_random_seed(SEED)

def make_env(rank: int):
    def _init():
        e = PipeRoutingEnvStep1(TRAIN_POOL, eval_mode=False, seed=SEED + rank)
        return Monitor(ActionMasker(e, lambda ee: ee.action_masks()), info_keywords=INFO_KEYS)
    return _init

venv = DummyVecEnv([make_env(i) for i in range(cfg.n_envs)])
assert is_masking_supported(venv), "action mask 미지원 — Layer 0 가 동작하지 않는다"

if cfg.gnarl_mode:
    POLICY, policy_kwargs = TrueGNARLPolicy, dict(net_arch=dict(pi=[64], vf=[64]))
else:
    POLICY = "MlpPolicy"
    policy_kwargs = dict(features_extractor_class=GNARLFeaturesExtractor,
                         features_extractor_kwargs=dict(features_dim=cfg.features_dim,
                                                        hidden=cfg.hidden, clip_mode=cfg.h_clip_mode),
                         net_arch=dict(pi=[128, 128], vf=[128, 128]))

def bc_pretrain(policy, n_batches, bs, lr):
    """C4 (§33) — A* 최적 행동 복제. 정책이 실제로 쓰는 logits = −(ΔJ + h(s'))/τ 에 교차 엔트로피를 건다.

    데이터는 순위 라벨 캐시의 학습 그룹(학습 풀 A* 경로 위 결정 시점)뿐이다 — 평가 풀은 쓰지 않는다.
    목표는 argmin t (t = ΔJ + A* 남은 J). 정확한 동점은 균등 분배 (A* 의 동점 선택은 전개 순서의 부산물).
    h_core 만 갱신한다 (logit 경로의 유일한 신경망, §6). τ · value head 는 건드리지 않는다.
    """
    assert RANK_TRAIN is not None, "BC 는 순위 라벨 캐시가 필요하다 (셀 7b)"
    ptr, t = RANK_TRAIN["ptr"], RANK_TRAIN["t"]
    G = len(ptr) - 1; L = int(np.diff(ptr).max())
    idx = np.full((G, L), -1, dtype=np.int64); tgt = np.zeros((G, L), dtype=np.float32)
    for g, (a, b) in enumerate(zip(ptr[:-1], ptr[1:])):
        idx[g, :b - a] = np.arange(a, b)
        tie = np.abs(t[a:b] - t[a:b].min()) < 1e-6
        tgt[g, :b - a] = tie / tie.sum()
    idx_t, tgt_t = torch.as_tensor(idx), torch.as_tensor(tgt)
    obs_t, dj_t = _RT["obs"], _RT["dj"]
    tau = _cur_tau(policy)
    dev = policy.device

    def _ce_top1(sel):
        ii = idx_t[sel]; valid = ii >= 0
        h = F.softplus(policy.h_core(obs_t[ii[valid]].to(dev))).squeeze(-1).cpu()
        q = torch.full(ii.shape, float("inf")); q[valid] = dj_t[ii[valid]] + h
        logp = torch.log_softmax(-q / tau, dim=1).masked_fill(~valid, 0.0)
        ce = -(tgt_t[sel] * logp).sum(1).mean()
        top1 = (tgt_t[sel].gather(1, q.argmin(1, keepdim=True)) > 0).float().mean()
        return ce, float(top1)

    def _all():
        with torch.no_grad():
            ce, top1 = _ce_top1(torch.arange(G))
        return float(ce), top1, h_rank_metrics(policy).get("rank_top1", float("nan"))

    ps = list(policy.h_core.parameters())
    opt = torch.optim.Adam(ps, lr=lr)
    rng = np.random.default_rng(SEED)
    ce0, tr0, ev0 = _all()
    curve = [(0, ce0, tr0, ev0)]
    print(f"BC (§33): 그룹 {G:,} (동점 {int((tgt > 0).sum(1).__gt__(1).sum())}) · τ {tau:.3f} · "
          f"{n_batches} 배치 × {bs} · lr {lr}")
    print(f"  배치    0 | CE {ce0:.4f} | 학습 그룹 top-1 {tr0:.3f} | 평가 풀 top-1 {ev0:.3f} (계기판)")
    for bi in range(n_batches):
        sel = torch.as_tensor(rng.integers(0, G, size=bs))
        ce, _ = _ce_top1(sel)
        opt.zero_grad(); ce.backward(); opt.step()
        if (bi + 1) % 500 == 0 or bi == n_batches - 1:
            c, tr, ev = _all()
            curve.append((bi + 1, c, tr, ev))
            print(f"  배치 {bi+1:4d} | CE {c:.4f} | 학습 그룹 top-1 {tr:.3f} | 평가 풀 top-1 {ev:.3f} (계기판)")
    PRETRAIN_INFO["bc"] = dict(n_batches=n_batches, bs=bs, lr=lr, tau=tau, groups=G, curve=curve)


# Colab 런타임이 끊겨도 Run All 만 다시 하면 이어서 학습한다 (셀 1b 의 작업 디렉터리 기준).
RESUME_PATH = cfg.resume_from
if RESUME_PATH is None and cfg.auto_resume and os.path.exists("checkpoint_step1_final.zip"):
    RESUME_PATH = "checkpoint_step1_final"
    print("자동 재개: checkpoint_step1_final.zip 발견 "
          "(처음부터 다시 하려면 cfg.auto_resume=False 로 두거나 파일을 지울 것)")

if RESUME_PATH:
    model = MaskablePPO.load(RESUME_PATH, env=venv, device="auto")
    START_STEPS = model.num_timesteps
    print(f"재개: {RESUME_PATH} 에서 {START_STEPS:,} 스텝부터 이어서 학습")
else:
    START_STEPS = 0
    model = MaskablePPO(POLICY, venv, learning_rate=cfg.learning_rate, n_steps=cfg.n_steps,
                        batch_size=cfg.batch_size, n_epochs=cfg.n_epochs, gamma=cfg.gamma,
                        clip_range=cfg.clip_range, ent_coef=cfg.ent_coef,
                        policy_kwargs=policy_kwargs, seed=SEED, verbose=0, device="auto")
    if H_EXTRACTOR_SD is not None:
        if cfg.gnarl_mode:
            model.policy.h_core.load_state_dict(H_EXTRACTOR_SD)
            print("사전학습 h(n) 주입 → TrueGNARLPolicy.h_core")
        else:
            for m_ in model.policy.modules():
                if isinstance(m_, GNARLFeaturesExtractor): m_.h_net.load_state_dict(H_EXTRACTOR_SD)
            print("사전학습 h(n) 주입 → GNARLFeaturesExtractor.h_net")
    if cfg.init_from:
        _src = MaskablePPO.load(cfg.init_from, device="cpu", print_system_info=False)
        _sd = zero_pad_state_dict(_src.policy.state_dict(), OBS_DIM)
        _own = model.policy.state_dict()
        _skip = [k_ for k_ in _sd if k_ in _own and _sd[k_].shape != _own[k_].shape]
        _bad = [k_ for k_ in _skip if not k_.startswith("mlp_extractor.")]
        assert not _bad, f"전이: mlp_extractor 밖에서 모양 불일치 {_bad}"
        _res = model.policy.load_state_dict({k_: v_ for k_, v_ in _sd.items() if k_ not in _skip}, strict=False)
        assert set(_res.missing_keys) == set(_skip) and not _res.unexpected_keys, _res
        PRETRAIN_INFO["init_from"] = dict(path=cfg.init_from, skipped=_skip,
                                          src_obs_dim=int(_src.observation_space.shape[0]))
        print(f"전이 (§38-2): {cfg.init_from} → 관측 {_src.observation_space.shape[0]} → {venv.observation_space.shape[0]} "
              f"· 건너뛴 키 {_skip}")
        del _src
    if cfg.gnarl_mode and cfg.bc_batches > 0:
        bc_pretrain(model.policy, cfg.bc_batches, cfg.bc_bs, cfg.bc_lr)

# sb3 는 reset_num_timesteps=False 일 때 total_timesteps 에 현재 스텝을 **더한다**.
# 그대로 5M 을 넘기면 재개할 때마다 5M 씩 늘어난다 → 남은 만큼만 넘긴다.
REMAINING_STEPS = max(0, cfg.total_timesteps - START_STEPS)

# ---- h_core 동결 (§19-2 h 드리프트 대응) ----
# 주의: 진짜 GNARL 에서 logits = -(g + ΔJ + h)/τ 이고 **h_core 가 logit 경로의 유일한 신경망**이다.
# 따라서 h 를 동결하면 정책에 남는 학습 파라미터는 τ 하나뿐이다 — PPO 는 순위를 바꾸지 못하고
# '사전학습 f 에 대한 확신의 세기' 만 조절한다. 이건 사실상 A* 휴리스틱 그리디 + 온도이고,
# 보상으로 정책을 배운다는 §1 철학과는 거리가 있다.
#   · 드리프트만 막고 학습은 유지하려면 → cfg.gnarl_unfreeze_at = 300_000 (워밍업 동결)
#   · τ 붕괴만 막으려면               → cfg.gnarl_tau_min = 0.05 (h 는 계속 학습)
def _set_h_trainable(flag: bool):
    if not cfg.gnarl_mode:
        return
    for _p in model.policy.h_core.parameters():
        _p.requires_grad_(flag)

if cfg.gnarl_mode and cfg.gnarl_freeze_h:
    _set_h_trainable(False)
    _n_free = sum(p.numel() for p in model.policy.parameters() if p.requires_grad)
    print(f"h_core 동결 — 학습 가능 파라미터 {_n_free:,} "
          f"(value head 제외하면 정책 쪽은 τ 1개뿐이다)")
    if cfg.gnarl_unfreeze_at:
        print(f"  → {cfg.gnarl_unfreeze_at:,} 스텝에서 자동 해제 (워밍업 동결)")
    else:
        print("  → 계속 동결. 정책 순위는 사전학습 h 에 고정된다 (위 주석 참고)")
if cfg.gnarl_mode and cfg.gnarl_tau_min > 0:
    print(f"τ 하한 {cfg.gnarl_tau_min} 적용 — 1차 런의 τ 붕괴(0.05→0.0117)를 막는다")

assert abs(model.gamma - cfg.gamma) < 1e-12, "§9 조건 2 위반 (γ_PBS ≠ γ_PPO)"
_npar = sum(p.numel() for p in model.policy.parameters() if p.requires_grad)
print(f"device={model.device} | 학습 파라미터 {_npar:,} | 관측 {venv.observation_space.shape[0]}차원 | "
      f"rollout {cfg.n_envs*cfg.n_steps} | 총 {cfg.total_timesteps:,} 스텝 "
      f"(이번 실행에서 {REMAINING_STEPS:,} 스텝)")
if REMAINING_STEPS == 0:
    print("이미 목표 스텝에 도달했다 — 셀 9 는 학습 없이 마지막 평가만 한다")

## 셀 9. 학습 루프

- **평가**: `cfg.eval_every` 마다 미학습 시나리오 48개로 측정 (success / length_ratio / deadlock / entropy / h MAE)
- **체크포인트**: `cfg.checkpoint_every`(기본 500K) 마다 스냅샷 + eval 최고점 갱신 시 `checkpoint_step1_best`
  + **평가할 때마다** 재개용 `checkpoint_step1_final` 갱신 (Colab 이 끊겨도 최대 100K 스텝만 잃는다)
- **재개**: 그냥 Run All 을 다시 하면 된다 (`cfg.auto_resume=True` 가 기본).
  특정 체크포인트에서 이어가려면 `cfg.resume_from = "checkpoint_step1_1500k"` 식으로 지정한다.
  처음부터 다시 하려면 `cfg.auto_resume=False` 로 두거나 `checkpoint_step1_final.zip` 을 지운다.

In [ ]:
#@title 9. 평가 + 콜백 + 학습
APPROACH_ZONE = 8   # 목표 진입 구간 = 맨해튼 8칸 이내 (§24-6, §25-2 방향 지표)

def _approach(dists) -> tuple:
    """(진입 구간 스텝, 재진입 횟수). 구역에 **처음 들어간 시점** 부터 끝까지를 센다 (§24-6 과 같은 규칙)."""
    d = np.asarray(dists); inz = d <= APPROACH_ZONE
    ent = int(np.argmax(inz)) if inz.any() else len(d) - 1
    return len(d) - 1 - ent, int(np.sum(inz[1:] & ~inz[:-1]))

def evaluate_routing(model, pool, deterministic: bool = True, seed: int = 0) -> dict:
    """평가 + 행동 패턴/entropy 계측 (§19-13: 50k→300k 하락 원인 분석용)."""
    env = PipeRoutingEnvStep1(pool, eval_mode=True, seed=seed)
    rng = np.random.default_rng(seed)
    rows = []
    ent, ent_unif, maxp, kinds, n_act, choice = [], [], [], [], [], []
    appr = []                                   # (에이전트 진입 스텝, A* 진입 스텝, 재진입)
    ovl = []                                    # A* 경로 겹침률 (§28-3)
    for i in range(len(pool)):
        obs, _ = env.reset(options={"index": i})
        done = False
        info = {}
        goal_a = np.asarray(env.goal, dtype=np.int64)
        dists = [int(np.abs(np.asarray(env.pos, dtype=np.int64) - goal_a).sum())]
        while not done:
            m = env.action_masks()
            cur_dir = env.dir
            if model is None:
                a = int(rng.choice(np.where(m)[0]))            # 랜덤 기준선
            else:
                # predict 대신 분포를 직접 받아 행동과 entropy 를 한 번에 얻는다 (추가 비용 없음)
                with torch.no_grad():
                    ot = torch.as_tensor(obs[None], device=model.device)
                    d = model.policy.get_distribution(ot, action_masks=m[None])
                    a = int(d.get_actions(deterministic=deterministic)[0])
                    pr = d.distribution.probs[0]
                    nv = int(m.sum())
                    # k 마스크 때문에 대부분 스텝은 직진 강제(유효행동 1개)다.
                    # 탐색을 재려면 **선택지가 2개 이상인 스텝**만 봐야 한다.
                    if nv >= 2:
                        ent.append(float(d.entropy()[0])); maxp.append(float(pr.max()))
                        ent_unif.append(float(np.log(nv)))
                n_act.append(nv)
                choice.append(int(m.sum() >= 2))
            # 행동 종류 분류
            if a == STAY: kinds.append("stay")
            elif a == cur_dir: kinds.append("straight")
            else:
                deg = float(TURN_DEG[cur_dir, a])
                kinds.append("turn45" if deg <= ELBOW45_MAX_DEG else
                             ("turn90" if deg <= ELBOW90_MAX_DEG else "sharp"))
            obs, r, term, trunc, info = env.step(a)
            dists.append(int(np.abs(np.asarray(env.pos, dtype=np.int64) - goal_a).sum()))
            done = term or trunc
        rows.append(info)
        sa, re_ = _approach(dists)
        b = env.base
        sb = (_approach([int(np.abs(np.asarray(p_, dtype=np.int64) - goal_a).sum()) for (p_, _, _, _) in b["states"]])[0]
              if b is not None else -1)
        appr.append((sa, sb, re_))
        # §28-3 A* 모방 감시: 에이전트 경로 칸 ↔ A* 최적 경로 칸 Jaccard (성공분에서만 집계)
        if b is not None:
            P = set(tuple(int(v) for v in q) for q in env.path)
            A = set(tuple(int(v) for v in q[0]) for q in b["states"])
            ovl.append(len(P & A) / max(len(P | A), 1))
        else:
            ovl.append(float("nan"))

    succ = np.array([r["success"] for r in rows])
    lr = np.array([r["length_ratio"] for r in rows], dtype=float)
    jr = np.array([r["j_ratio"] for r in rows], dtype=float)
    nb = np.array([r["n_bends"] for r in rows], dtype=float)
    nb0 = np.array([r["bends_base"] for r in rows], dtype=float)
    m_ok = succ > 0.5
    f = lambda v: float(np.mean(v)) if len(v) else float("nan")
    er = np.array([r["elbow_ratio"] for r in rows], dtype=float)
    ja = np.array([r["j_ratio_all"] for r in rows], dtype=float)
    # §7 보완 (2026-09-23) — 성공분 기준은 **어려운 시나리오를 포기해도** 만족될 수 있다.
    # 그래서 평가셋을 A* 기준선 J 로 3분위(쉬움/보통/어려움)로 나눠 구간별 success·J_ratio 를 따로 본다.
    # 경계는 값이 아니라 **순위**로 나눠 구간 크기를 같게 한다 (48 → 16/16/16, 동점에도 안정적).
    bj = np.array([pool[i]["astar"][pool[i]["dirs"][0]]["J"] for i in range(len(rows))], dtype=float)
    tier = np.empty(len(rows), dtype=int)
    tier[np.argsort(bj, kind="mergesort")] = np.arange(len(rows)) * 3 // len(rows)   # 0 쉬움 1 보통 2 어려움
    tiers = {}
    for t, nm in enumerate(("easy", "mid", "hard")):
        sel = tier == t
        tiers[f"succ_{nm}"] = float(succ[sel].mean()) if sel.any() else float("nan")
        tiers[f"jr_{nm}"] = f(jr[sel & m_ok])
    tiers["hard_gap"] = float(succ.mean() - tiers["succ_hard"])
    tiers["alarm_hard"] = float(tiers["hard_gap"] >= cfg.alarm_hard_gap)
    return dict(
        success_rate=float(succ.mean()),
        deadlock_rate=float(np.mean([r["deadlock"] for r in rows])),
        timeout_rate=float(np.mean([r["timeout"] for r in rows])),
        length_ratio=f(lr[m_ok]),                  # 참고값 — J 의 부분(직관 길이)만 잰다
        j_ratio=f(jr[m_ok]),                       # §7 판정: 성공 에피소드 기준
        elbow_ratio=f(er[m_ok & ~np.isnan(er)]),   # §7 판정: 성공 에피소드 기준 (기준선 엘보 0 은 제외)
        j_ratio_all=f(ja[~np.isnan(ja)]),          # 계기판: 전체 에피소드, 실패는 '쓴 J + 남은 최소 J'
        **tiers,
        # §25-2 방향 지표 — 목표 8칸 구역 첫 진입부터 도달까지 (성공분). h1k 300K: 38.9 vs A* 3.9
        steps_approach=f(np.array([x[0] for x in appr], dtype=float)[m_ok]),
        steps_approach_base=f(np.array([x[1] for x in appr], dtype=float)[m_ok]),
        reentries=f(np.array([x[2] for x in appr], dtype=float)[m_ok]),
        reentries_fail=f(np.array([x[2] for x in appr], dtype=float)[~m_ok]),
        astar_overlap=f(np.array(ovl, dtype=float)[m_ok]),   # §28-3 — 성공분 Jaccard · 기록 전용 (감시 아님, §39)
        bends_agent=f(nb[m_ok]),
        bends_astar=f(nb0[m_ok]),
        ep_steps=float(np.mean([r["ep_steps"] for r in rows])),
        n_invalid=float(np.mean([r["n_invalid"] for r in rows])),
        # --- 행동 패턴 / 탐색 계측 ---
        entropy=float(np.mean(ent)) if ent else float("nan"),
        entropy_ratio=float(np.mean(ent) / np.mean(ent_unif)) if ent else float("nan"),  # 1.0=균등
        max_prob=float(np.mean(maxp)) if maxp else float("nan"),
        n_valid=float(np.mean(n_act)) if n_act else float("nan"),
        choice_rate=float(np.mean(choice)) if choice else float("nan"),   # 선택지 ≥2 인 스텝 비율
        n_valid_at_choice=float(np.mean([x for x in n_act if x >= 2])) if any(x >= 2 for x in n_act) else float("nan"),
        **{f"act_{k}": float(np.mean([x == k for x in kinds])) for k in
           ("straight", "turn45", "turn90", "sharp", "stay")},
    )

def fmt_metrics(m: dict) -> str:
    return (f"success {m['success_rate']:.3f} | j_ratio {m['j_ratio']:.3f} | "
            f"elbow_ratio {m.get('elbow_ratio', float('nan')):.2f} | length_ratio {m['length_ratio']:.3f} | "
            f"bends {m['bends_agent']:.1f} vs A* {m['bends_astar']:.1f} | "
            f"deadlock {m['deadlock_rate']:.3f} | timeout {m['timeout_rate']:.3f} | "
            f"steps {m['ep_steps']:.0f}"
            + (f" | ent {m['entropy']:.2f}({m['entropy_ratio']:.2f}) maxp {m['max_prob']:.2f} "
               f"choice {m['choice_rate']:.2f}"
               if m.get("entropy") == m.get("entropy") else "")
            + (f" | approach {m['steps_approach']:.1f} vs A* {m['steps_approach_base']:.1f}"
               f" ovl {m.get('astar_overlap', float('nan')):.2f}"
               if "steps_approach" in m else "")
            + (f" || J_all {m['j_ratio_all']:.2f} | tier(e/m/h) {m['succ_easy']:.2f}/{m['succ_mid']:.2f}/"
               f"{m['succ_hard']:.2f}" + (" [ALARM hard]" if m.get("alarm_hard") else "")
               if "succ_hard" in m else "")
            + (f" || train success {m['train_success']:.3f}" if "train_success" in m else ""))

# 랜덤(마스킹된) 정책 기준선 — 학습이 무엇을 이겨야 하는지의 바닥값
_rand = evaluate_routing(None, EVAL_POOL)
print("랜덤 정책            :", fmt_metrics(_rand))

# Layer 0 확장 옵션의 효과 측정: 1-ply 데드락 회피 마스크를 켜면 데드락이 얼마나 줄어드나
_prev = cfg.deadlock_1ply
cfg.deadlock_1ply = not _prev
print(f"랜덤 + 1ply={cfg.deadlock_1ply}   :", fmt_metrics(evaluate_routing(None, EVAL_POOL)))
cfg.deadlock_1ply = _prev
print(f"\n현재 설정: deadlock_1ply={cfg.deadlock_1ply}")
print("주: 랜덤 정책의 deadlock_rate 가 1.0 이어도 정상이다. k=6 마스크 하에서 무작정 직진하면"
      " 벽 앞에서 s<k 라 꺾지 못해 막힌다. 학습은 '미리 꺾는' 정책을 찾는 것이고,"
      " A* 가 전 시나리오를 푼다는 사실이 해가 존재함을 보증한다 (§7 도달성 검증).")


from stable_baselines3.common.callbacks import BaseCallback

# ---- h(n) 품질 감시용 라벨 (평가 시나리오의 A* 접미 비용 — 학습에는 쓰지 않는다) ----
def build_h_probe(pool, max_states=3000):
    X, Y = [], []
    for sc in pool:
        for d, sol in sc["astar"].items():
            Jt = sol["J"]
            for (pos, dd, ss, gg) in sol["states"]:
                X.append(build_obs(sc["pad"], pos, dd, ss, gg, sc["goal"], gravity=is_grav(sc))); Y.append((Jt - gg) / J_MAX)
    X = np.stack(X); Y = np.asarray(Y, dtype=np.float32)[:, None]
    if len(X) > max_states:
        sel = np.random.default_rng(0).choice(len(X), max_states, replace=False)
        X, Y = X[sel], Y[sel]
    return torch.as_tensor(X), torch.as_tensor(Y)

H_PROBE_X, H_PROBE_Y = build_h_probe(EVAL_POOL)

def h_predict(model, X):
    """두 구조 모두에서 h(n) 예측을 꺼낸다 (정규화 J 단위)."""
    with torch.no_grad():
        x = X.to(model.device)
        p = model.policy
        if cfg.gnarl_mode:
            return F.softplus(p.h_core(x)).cpu()
        for m_ in p.modules():
            if isinstance(m_, GNARLFeaturesExtractor):
                return m_.h_raw(x).cpu()
    return torch.zeros(len(X), 1)

def h_mae_kg(model):
    return float((h_predict(model, H_PROBE_X) - H_PROBE_Y).abs().mean()) * J_MAX

HISTORY, BEST = [], {"success_rate": -1.0, "timesteps": 0}

class Step1Callback(BaseCallback):
    def __init__(self, pool, verbose=1):
        super().__init__(verbose)
        self.pool = pool; self._last_eval = 0; self._last_ckpt = 0

    def _on_training_start(self) -> None:
        # 재개 시 num_timesteps 는 0 이 아니다. 0 에서 재면 첫 스텝에 평가/저장이 터진다.
        self._last_eval = self._last_ckpt = self.num_timesteps

    def _log(self):
        m = evaluate_routing(self.model, self.pool)
        m["timesteps"] = int(self.num_timesteps)
        m["h_mae_kg"] = h_mae_kg(self.model)
        if cfg.h_rank_metrics:
            m.update(h_rank_metrics(self.model.policy))     # §21-5 계기판
        if "rank_top1" in m:                                # §23-4 경보 (판정 아님)
            m["alarm_rank"] = float(m["rank_top1"] < cfg.alarm_top1 or m["rank_spearman"] < cfg.alarm_rho)
        if cfg.gnarl_mode:
            # τ 와 h MAE 는 §19 의 두 붕괴 기전을 직접 보는 계기판이다. 반드시 같이 남긴다.
            m["tau"] = float(self.model.policy.log_tau.detach().exp())
        if self.model.ep_info_buffer:
            m["train_ep_rew_mean"] = float(np.mean([e["r"] for e in self.model.ep_info_buffer]))
            for key in ("success", "deadlock"):
                v = [e[key] for e in self.model.ep_info_buffer if key in e]
                if v: m[f"train_{key}"] = float(np.mean(v))
        HISTORY.append(m)
        for k_, v in m.items():
            if k_ != "timesteps": self.logger.record(f"eval/{k_}", v)
        if USE_WANDB and run is not None:
            wandb.log({f"eval/{k_}": v for k_, v in m.items() if k_ != "timesteps"},
                      step=self.num_timesteps)
        if m["success_rate"] > BEST["success_rate"]:
            BEST.update(m); self.model.save("checkpoint_step1_best")
            print(f"      ★ 최고점 갱신 {m['success_rate']:.3f} → checkpoint_step1_best.zip")
        # 재개용 파일은 매 평가마다 갱신한다 (Colab 이 끊겨도 최대 eval_every 만큼만 잃는다)
        self.model.save("checkpoint_step1_final")
        json.dump(dict(history=HISTORY, best=BEST, cfg=asdict(cfg)),
                  open("step1_history.json", "w"), ensure_ascii=False, indent=1, default=float)
        if self.verbose:
            print(f"[{self.num_timesteps:>9,}] {fmt_metrics(m)} | h MAE {m['h_mae_kg']:.2f} kg"
                  + (f" | tau {m['tau']:.4f}" if "tau" in m else "")
                  + (f" | top1 {m['rank_top1']:.3f} rho {m['rank_spearman']:+.3f} "
                     f"|dh| {m['h_spread']:.4f}" if "rank_top1" in m else "")
                  + (" [ALARM rank]" if m.get("alarm_rank") else ""))
        return m

    def _on_step(self) -> bool:
        # 예약된 h_core 해제 (워밍업 동결)
        if (cfg.gnarl_mode and cfg.gnarl_freeze_h and cfg.gnarl_unfreeze_at
                and self.num_timesteps >= cfg.gnarl_unfreeze_at
                and not next(self.model.policy.h_core.parameters()).requires_grad):
            _set_h_trainable(True)
            print(f"      h_core 동결 해제 @ {self.num_timesteps:,} 스텝")
        if self.num_timesteps - self._last_ckpt >= cfg.checkpoint_every:
            self._last_ckpt = self.num_timesteps
            self.model.save(f"checkpoint_step1_{self.num_timesteps//1000}k")
            print(f"      체크포인트 저장: checkpoint_step1_{self.num_timesteps//1000}k.zip")
        if self.num_timesteps - self._last_eval >= cfg.eval_every:
            self._last_eval = self.num_timesteps
            self._log()
        return True

    def _on_training_end(self) -> None:
        self._log()

print("랜덤 정책 기준선:", fmt_metrics(evaluate_routing(None, EVAL_POOL)))
print(f"사전학습 직후 h MAE: {h_mae_kg(model):.2f} kg (평가 시나리오 라벨 기준)\n")

import importlib.util as _ilu
_progress = _ilu.find_spec("rich") is not None
callbacks = [Step1Callback(EVAL_POOL)]
if cfg.h_rank_batches > 0:
    callbacks.append(HRankCallback())     # §21-3 PPO 본체는 건드리지 않는다
if USE_WANDB and run is not None:
    callbacks.append(WandbCallback(gradient_save_freq=0, verbose=0))

_t0 = time.time()
if REMAINING_STEPS > 0:
    model.learn(total_timesteps=REMAINING_STEPS, callback=callbacks,
                reset_num_timesteps=not bool(RESUME_PATH), progress_bar=_progress)
else:
    # 이미 목표 스텝에 도달한 경우: 학습 없이 마지막 평가만 남긴다.
    _m = evaluate_routing(model, EVAL_POOL)
    _m["timesteps"] = int(model.num_timesteps); _m["h_mae_kg"] = h_mae_kg(model)
    if cfg.h_rank_metrics:
        _m.update(h_rank_metrics(model.policy))
    HISTORY.append(_m)
    if _m["success_rate"] > BEST["success_rate"]: BEST.update(_m)
    print(f"[{model.num_timesteps:>9,}] {fmt_metrics(_m)} | h MAE {_m['h_mae_kg']:.2f} kg")
model.save("checkpoint_step1_final")
print(f"\n학습 완료: {(time.time()-_t0)/60:.1f}분 | {model.num_timesteps:,} 스텝 | "
      f"checkpoint_step1_final.zip 저장")

## 셀 10. 시각화 (CLAUDE.md §17 명세)

§17-1 학습곡선 · §17-2 경로품질 · §17-3 에피소드 3D · §17-4 h 품질 · §17-5 f 분포 · §17-6 졸업 판정

> 그래프 라벨은 ASCII 다 — Colab 기본 matplotlib 에 한글 폰트가 없어 □ 로 깨진다.

In [ ]:
#@title 10. 대시보드 + 졸업 판정
import matplotlib
try:
    get_ipython()          # Colab/Jupyter → 기본 백엔드 유지 (Agg 로 바꾸면 그림이 안 보인다)
except NameError:
    matplotlib.use("Agg")  # 순수 스크립트 → 파일로만 저장
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

SURF, INK, INK2, MUTED = "#fcfcfb", "#0b0b0b", "#52514e", "#8a8983"
C_EVAL, C_TRAIN, C_OK, C_WARN = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"

def _ax(a):
    a.set_facecolor(SURF)
    for s in ("top", "right"): a.spines[s].set_visible(False)
    for s in ("left", "bottom"): a.spines[s].set_color(MUTED)
    a.grid(axis="y", color=MUTED, alpha=.22, lw=.8); a.set_axisbelow(True)
    a.tick_params(colors=INK2, labelsize=9)

def plot_dashboard(save="step1_dashboard.png"):
    """§17-1 학습곡선 + §17-2 경로품질."""
    if not HISTORY:
        print("HISTORY 가 비었다 — 셀 9 를 먼저 실행할 것"); return
    ts = [h["timesteps"] for h in HISTORY]
    fig, ax = plt.subplots(1, 4, figsize=(19, 4.2), facecolor=SURF)
    for a in ax: _ax(a)
    # §17-1
    ax[0].plot(ts, [h["success_rate"] for h in HISTORY], color=C_EVAL, lw=2, marker="o", ms=5,
               mec=SURF, mew=1.5, label="eval")
    ax[0].plot(ts, [h.get("train_success", np.nan) for h in HISTORY], color=C_TRAIN, lw=2, ls="--",
               marker="s", ms=4, mec=SURF, mew=1.2, label="train")
    ax[0].axhline(1.0, color=MUTED, ls="-", lw=1.2)
    ax[0].annotate("A* baseline = 1.0", (ts[-1], 1.0), xytext=(-4, -12), textcoords="offset points",
                   ha="right", color=INK2, fontsize=8.5)
    ax[0].axhline(cfg.grad_success, color="#e34948", ls=":", lw=1.6)
    ax[0].annotate(f"graduation {cfg.grad_success}", (ts[0], cfg.grad_success), xytext=(2, 4),
                   textcoords="offset points", color="#e34948", fontsize=8.5)
    ax[0].set_ylim(0, 1.05); ax[0].set_title("17-1  success_rate", color=INK, fontsize=11, loc="left")
    ax[0].legend(frameon=False, fontsize=9, labelcolor=INK2)
    # §17-2
    # §7 (2026-09-23): 경로 품질은 J_ratio 와 elbow_ratio 로 판정한다 (성공 에피소드 기준)
    ax[1].plot(ts, [h.get("j_ratio_all", np.nan) for h in HISTORY], color=C_TRAIN, lw=1.5, ls="--",
               label="all episodes (dashboard)")
    ax[1].axhline(cfg.target_j_ratio, color=C_OK, ls=":", lw=1.4)
    ax[1].annotate(f"goal {cfg.target_j_ratio}", (ts[-1], cfg.target_j_ratio), xytext=(-4, -12),
                   textcoords="offset points", ha="right", color=C_OK, fontsize=8.5)
    for a, key, lab, thr in ((ax[1], "j_ratio", "17-2a  J_ratio (success)", cfg.grad_j_ratio),
                             (ax[2], "deadlock_rate", "17-2b  deadlock rate", cfg.grad_deadlock),
                             (ax[3], "elbow_ratio", "17-2c  elbow_ratio (success)", cfg.grad_elbow_ratio)):
        a.plot(ts, [h.get(key, np.nan) for h in HISTORY], color=C_EVAL, lw=2, marker="o", ms=5,
               mec=SURF, mew=1.5)
        a.axhline(thr, color="#e34948", ls=":", lw=1.6)
        a.annotate(f"target {thr}", (ts[0], thr), xytext=(2, 4), textcoords="offset points",
                   color="#e34948", fontsize=8.5)
        a.set_title(lab, color=INK, fontsize=11, loc="left")
    ax[1].legend(frameon=False, fontsize=8, labelcolor=INK2)
    for a in ax: a.set_xlabel("timesteps", color=INK2, fontsize=9)
    fig.tight_layout(); fig.savefig(save, dpi=130, facecolor=SURF); plt.show()
    print(f"→ {save}")

def plot_episodes(indices=(0, 1, 2), save="step1_routes.png"):
    """§17-3 에피소드 3D — 파이프 경로/장애물/시작·목표/A* 최적경로."""
    env = PipeRoutingEnvStep1(EVAL_POOL, eval_mode=True)
    fig = plt.figure(figsize=(5.2 * len(indices), 4.8), facecolor=SURF)
    for j, idx in enumerate(indices):
        sc = EVAL_POOL[idx]
        obs, _ = env.reset(options={"index": idx}); done = False; info = {}
        while not done:
            m = env.action_masks()
            with torch.no_grad():
                a = int(model.policy.get_distribution(
                    torch.as_tensor(obs[None], device=model.device),
                    action_masks=m[None]).get_actions(deterministic=True)[0])
            obs, r, te, tr, info = env.step(a); done = te or tr
        a3 = fig.add_subplot(1, len(indices), j + 1, projection="3d")
        ob = np.argwhere(~sc["free"])
        if len(ob):
            sub = ob[:: max(1, len(ob) // 900)]
            a3.scatter(sub[:, 0], sub[:, 1], sub[:, 2], s=5, c="0.75", alpha=.20, marker="s")
        ap = np.array([st[0] for st in sc["astar"][sc["dirs"][0]]["states"]])
        a3.plot(ap[:, 0], ap[:, 1], ap[:, 2], color=C_WARN, lw=2, ls="--",
                label=f"A* J={sc['astar'][sc['dirs'][0]]['J']:.1f}kg")
        pp = np.array(env.path)
        a3.plot(pp[:, 0], pp[:, 1], pp[:, 2], color=C_EVAL, lw=2.2, label=f"agent J={info['j_agent']:.1f}kg")
        a3.scatter(*sc["start"], c=C_OK, s=70, marker="o")
        a3.scatter(*sc["goal"], c="#e34948", s=90, marker="*")
        a3.set_xlim(0, cfg.NX); a3.set_ylim(0, cfg.NY); a3.set_zlim(0, cfg.NZ)
        ok = info["success"] > .5
        a3.set_title(f"#{idx}  J={info['j_agent']:.1f}kg  elbows={int(info['n_bends'])}  "
                     f"{'SUCCESS' if ok else ('DEADLOCK' if info['deadlock']>.5 else 'TIMEOUT')}",
                     fontsize=10, color=INK)
        a3.legend(fontsize=7.5, loc="upper left")
    fig.tight_layout(); fig.savefig(save, dpi=130, facecolor=SURF); plt.show()
    print(f"→ {save}")

def plot_h_and_f(save="step1_h_f.png"):
    """§17-4 h 예측 vs 실제 남은 J · §17-5 이웃 f 분포 (g/h 분리)."""
    fig, ax = plt.subplots(1, 3, figsize=(15, 4.2), facecolor=SURF)
    for a in ax: _ax(a)
    # 17-4
    pred = h_predict(model, H_PROBE_X).numpy().ravel() * J_MAX
    true = H_PROBE_Y.numpy().ravel() * J_MAX
    s = np.random.default_rng(0).choice(len(pred), min(1500, len(pred)), replace=False)
    ax[0].scatter(true[s], pred[s], s=9, color=C_EVAL, alpha=.35, edgecolor="none")
    lim = [0, max(true.max(), pred.max()) * 1.05]
    ax[0].plot(lim, lim, color=MUTED, ls="--", lw=1.4)
    ax[0].set_xlabel("actual remaining J (kg)", color=INK2, fontsize=9)
    ax[0].set_ylabel("predicted h (kg)", color=INK2, fontsize=9)
    ax[0].set_title(f"17-4  h quality  MAE={np.abs(pred-true).mean():.2f} kg",
                    color=INK, fontsize=11, loc="left")
    # 17-5
    env = PipeRoutingEnvStep1(EVAL_POOL, eval_mode=True)
    obs, _ = env.reset(options={"index": 0}); F_, G_, H_ = [], [], []
    for t in range(60):
        m = env.action_masks()
        if cfg.gnarl_mode:
            ot = torch.as_tensor(obs[None], device=model.device)
            with torch.no_grad():
                own, nbr, dJ = model.policy._split(ot)
                h = model.policy.h_admissible(nbr.reshape(-1, OBS_DIM)).reshape(-1).cpu().numpy()
                f = model.policy.f_values(ot)[0].cpu().numpy()
            gpart = (float(own[0, 33]) + dJ[0].cpu().numpy())
            F_ += list(f[m]); G_ += list(gpart[m]); H_ += list(h[m])
            a = int(np.argmin(np.where(m, f, np.inf)))
        else:
            a, _ = model.predict(obs, action_masks=m, deterministic=True); a = int(a)
        obs, r, te, tr, _ = env.step(a)
        if te or tr: break
    if F_:
        ax[1].hist(F_, bins=28, color=C_EVAL, alpha=.85)
        ax[1].set_title("17-5a  f = g + dJ + h  (valid neighbours)", color=INK, fontsize=11, loc="left")
        ax[1].set_xlabel("f (normalized J)", color=INK2, fontsize=9)
        ax[2].hist(G_, bins=28, color=C_WARN, alpha=.75, label="g + dJ")
        ax[2].hist(H_, bins=28, color=C_OK, alpha=.75, label="h")
        ax[2].legend(frameon=False, fontsize=9, labelcolor=INK2)
        ax[2].set_title("17-5b  g / h split", color=INK, fontsize=11, loc="left")
        ax[2].set_xlabel("normalized J", color=INK2, fontsize=9)
    else:
        for a in (ax[1], ax[2]): a.set_title("17-5  (gnarl_mode=False 에서는 f 가 없다)",
                                             color=INK2, fontsize=10, loc="left")
    fig.tight_layout(); fig.savefig(save, dpi=130, facecolor=SURF); plt.show()
    print(f"→ {save}")

def graduation_check(m=None, save_on_pass=True):
    """§17-6 졸업 판정 — 4개 기준 각각 + 전체."""
    m = m or (HISTORY[-1] if HISTORY else None)
    if m is None:
        print("평가 기록이 없다"); return False
    # §7 (2026-09-22): h 지표는 졸업 기준에서 제외했다. MAE 는 대부분이 상수 편향이라
    # 판정 기준으로 무효임이 §20-4 에서 실측됐다 (raw 25.1kg 인데 success 0.19).
    # §7 (2026-09-23 개정): 4기준. J_ratio·elbow_ratio 는 **성공 에피소드 기준**이다.
    rows = [("eval success_rate", m["success_rate"], cfg.grad_success, "≥"),
            ("J_ratio (success)", m["j_ratio"], cfg.grad_j_ratio, "≤"),
            ("deadlock rate",     m["deadlock_rate"], cfg.grad_deadlock, "≤"),
            ("elbow_ratio (succ)", m.get("elbow_ratio", float("nan")), cfg.grad_elbow_ratio, "≤")]
    print("=" * 62); print(f"Step 1 졸업 판정  @ {m['timesteps']:,} 스텝"); print("=" * 62)
    print(f"{'기준':<22}{'측정값':>10}{'목표':>10}   판정")
    allok = True
    for name, val, thr, op in rows:
        ok = (val >= thr) if op == "≥" else (val <= thr)
        ok = bool(ok) and val == val
        allok &= ok
        print(f"{name:<22}{val:>10.3f}{op+str(thr):>10}   {'PASS' if ok else 'FAIL'}")
    print("-" * 62)
    # 경로 품질 계기판 (판정 아님 — §7)
    print(f"{'[목표선] J_ratio':<22}{m['j_ratio']:>10.3f}{'≤'+str(cfg.target_j_ratio):>10}   "
          f"{'도달' if m['j_ratio'] <= cfg.target_j_ratio else '미도달'} (졸업 후 개선 대상)")
    if "j_ratio_all" in m:
        print(f"{'[계기판] J_ratio 전체':<22}{m['j_ratio_all']:>10.3f}{'참고':>10}   (실패 = 쓴 J + 남은 최소 J)")
    print(f"{'[계기판] length_ratio':<22}{m['length_ratio']:>10.3f}{'참고':>10}")
    if "succ_hard" in m:
        print(f"{'[계기판] 난이도 3분위':<22}  success 쉬움 {m['succ_easy']:.3f} / 보통 {m['succ_mid']:.3f} / "
              f"어려움 {m['succ_hard']:.3f}")
        print(f"{'':<22}  J_ratio 쉬움 {m['jr_easy']:.3f} / 보통 {m['jr_mid']:.3f} / 어려움 {m['jr_hard']:.3f}")
        print(f"{'':<22}  어려움 격차 {m['hard_gap']:+.3f} (경보 ≥ {cfg.alarm_hard_gap})  "
              f"→ {'⚠ 경보 — 어려운 시나리오를 포기하고 있다' if m['alarm_hard'] else '정상'}")
    # h 계기판 (판정 아님 — §21-5)
    print(f"{'[계기판] h MAE (kg)':<22}{m.get('h_mae_kg', float('nan')):>10.3f}{'참고':>10}")
    for nm, key, f_ in (("[계기판] top-1 일치율", "rank_top1", "{:>10.3f}"),
                        ("[계기판] Spearman rho", "rank_spearman", "{:>10.3f}"),
                        ("[계기판] 이웃 |dh|", "h_spread", "{:>10.4f}")):
        if key in m:
            print(f"{nm:<22}" + f_.format(m[key]) + f"{'참고':>10}")
    if "rank_top1" in m:
        al = m["rank_top1"] < cfg.alarm_top1 or m["rank_spearman"] < cfg.alarm_rho
        print(f"{'[계기판] h 순위 경보':<22}  top-1 < {cfg.alarm_top1} 또는 rho < {cfg.alarm_rho}  "
              f"→ {'⚠ 경보' if al else '정상'}")
    print("-" * 62); print("전체:", "졸업" if allok else "미달")
    if allok and save_on_pass:
        model.save("checkpoint_step1_graduated")
        print("→ checkpoint_step1_graduated.zip 저장")
    if not allok:
        print("→ 셀 12 autoresearch 로 넘어가거나, CLAUDE.md §13 G3-L1(관측 설계)을 검토할 것")
    return allok

plot_dashboard(); plot_episodes(); plot_h_and_f()
GRADUATED = graduation_check()

## 셀 11. 체크포인트 관리 (CLAUDE.md §18)

| 파일 | 내용 |
|---|---|
| `checkpoint_step1_best.pt` | eval 최고점 시점 |
| `checkpoint_step1_final.pt` | 학습 완료 시점 |
| `checkpoint_step1_<N>k.zip` | 500K 마다 (재개용 sb3 형식) |

`.pt` 는 **state_dict + optimizer + cfg + 성능 수치**를 담는다 (사람이 열어보는 용도).
**재개는 sb3 `.zip` 으로 한다** — `cfg.resume_from = "checkpoint_step1_final"`.

In [ ]:
#@title 11. 체크포인트 저장/조회
def save_ckpt_pt(path, metrics, note=""):
    torch.save(dict(
        policy_state_dict=model.policy.state_dict(),
        optimizer_state_dict=model.policy.optimizer.state_dict(),
        cfg=asdict(cfg), metrics=dict(metrics), note=note,
        num_timesteps=int(model.num_timesteps),
        pretrain=dict(PRETRAIN_INFO),
        baseline_file=BASELINE_PATH, scenario_cache=CACHE,
        saved_at=time.strftime("%Y-%m-%d %H:%M:%S"),
    ), path)
    sz = os.path.getsize(path) / 1e6
    print(f"  {path}  ({sz:.1f} MB)  @ {model.num_timesteps:,} 스텝  "
          f"success={metrics.get('success_rate', float('nan')):.3f}")

save_ckpt_pt("checkpoint_step1_final.pt", HISTORY[-1] if HISTORY else {}, "학습 완료 시점")
save_ckpt_pt("checkpoint_step1_best.pt", BEST, f"eval 최고점 @ {BEST.get('timesteps',0):,}")

print("\n현재 디렉터리의 체크포인트:")
for f in sorted(x for x in os.listdir(".") if x.startswith("checkpoint_step1")):
    print(f"  {f:<42} {os.path.getsize(f)/1e6:>6.1f} MB")
print(f"\n최고점: success {BEST.get('success_rate', float('nan')):.3f} "
      f"@ {BEST.get('timesteps', 0):,} 스텝")
print("재개 방법: 같은 노트북을 Run All (셀 8 이 checkpoint_step1_final.zip 을 자동으로 이어받는다)")

json.dump(dict(history=HISTORY, best=BEST, cfg=asdict(cfg), pretrain=dict(PRETRAIN_INFO),
               graduated=bool(GRADUATED)),
          open("step1_history.json", "w"), ensure_ascii=False, indent=1, default=float)
print("→ step1_history.json 저장")

## 셀 12. autoresearch (졸업 미달 시에만)

CLAUDE.md §10 이 허용한 범위만 탐색한다 — **Layer 2 Φ 계수와 PPO 하이퍼파라미터뿐**.
물리 상수(J)는 탐색 대상이 아니다.

각 trial 은 **300K 스텝**(약 11분)이라 `n_trials=8` 이면 약 1.5시간이다.
끝나면 최적 cfg 를 적용해 5M 재학습을 돌린다 (`RERUN_AFTER_SEARCH=True` 일 때).

> **미결:** G3 의 ablation 60런에서 이 범위의 손잡이들은 전부 천장을 못 넘었다 (§16-2).
> 격자가 달라졌으므로 다시 재보는 것이지, 효과가 있다는 근거가 있어서가 아니다.

In [ ]:
#@title 12. Optuna 탐색
RUN_AUTORESEARCH = False   #@param {type:"boolean"}
RERUN_AFTER_SEARCH = False #@param {type:"boolean"}
N_TRIALS = 8               #@param {type:"integer"}
TRIAL_STEPS = 300_000      #@param {type:"integer"}

def _trial_run(params, steps):
    old = {k: getattr(cfg, k) for k in params}
    for k, v in params.items(): setattr(cfg, k, v)
    try:
        v_ = DummyVecEnv([make_env(1000 + i) for i in range(cfg.n_envs)])
        m_ = MaskablePPO(POLICY, v_, learning_rate=cfg.learning_rate, n_steps=cfg.n_steps,
                         batch_size=cfg.batch_size, n_epochs=cfg.n_epochs, gamma=cfg.gamma,
                         clip_range=cfg.clip_range, ent_coef=cfg.ent_coef,
                         policy_kwargs=policy_kwargs, seed=SEED, verbose=0, device="auto")
        if H_EXTRACTOR_SD is not None and cfg.gnarl_mode:
            m_.policy.h_core.load_state_dict(H_EXTRACTOR_SD)
        m_.learn(total_timesteps=steps, progress_bar=False)
        res = evaluate_routing(m_, EVAL_POOL)
    finally:
        for k, v in old.items(): setattr(cfg, k, v)
    return res

if RUN_AUTORESEARCH and not GRADUATED:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    def objective(trial):
        p = dict(
            phi_goal_ratio=trial.suggest_float("phi_goal_ratio", 0.6, 1.0),
            phi_cong_ratio=trial.suggest_float("phi_cong_ratio", 0.0, 0.4),
            learning_rate=trial.suggest_float("learning_rate", 1e-4, 5e-4, log=True),
            n_steps=trial.suggest_categorical("n_steps", [256, 512, 1024, 2048]),
        )
        p["phi_type"] = "combined" if p["phi_cong_ratio"] > 0.05 else "manhattan"
        r = _trial_run(p, TRIAL_STEPS)
        print(f"  trial {trial.number}: success {r['success_rate']:.3f} | {p}")
        return r["success_rate"]

    study = optuna.create_study(direction="maximize", study_name="step1")
    study.optimize(objective, n_trials=N_TRIALS)
    print("\n최적 파라미터:", study.best_params, "| success", round(study.best_value, 3))
    json.dump(dict(best_params=study.best_params, best_value=study.best_value,
                   trials=[dict(number=t.number, params=t.params, value=t.value)
                           for t in study.trials]),
              open("step1_autoresearch.json", "w"), ensure_ascii=False, indent=1)
    print("→ step1_autoresearch.json 저장")

    if RERUN_AFTER_SEARCH:
        for k, v in study.best_params.items(): setattr(cfg, k, v)
        cfg.phi_type = "combined" if cfg.phi_cong_ratio > 0.05 else "manhattan"
        print(f"\n최적 cfg 적용 후 {cfg.total_timesteps:,} 스텝 재학습 — 셀 8~11 을 다시 실행할 것")
elif GRADUATED:
    print("졸업 기준을 충족했다 — autoresearch 불필요")
else:
    print("건너뜀 (RUN_AUTORESEARCH=False)")

---

## 다음 세션에서 할 일

1. **이 노트북을 Run All** (셀 1~11). 5M 학습은 약 3시간.
2. 셀 10 의 졸업 판정 표를 본다.
   - **졸업** → `checkpoint_step1_graduated.zip` 으로 Step 2(벤딩 반경 + 엘보 J) 전이 착수
   - **미달** → 셀 12 `RUN_AUTORESEARCH=True` 또는 CLAUDE.md §13 `G3-L1`(관측 설계) 검토
3. 결과 수치를 CLAUDE.md §15·§16 에 기록한다 (§16-4 측정 프로토콜: **4~6시드 + 산포**).

**중단되면**: 같은 노트북을 **Run All** 만 다시 한다 (셀 8 이 자동으로 이어받는다).

*추측은 "추측:", 미결은 "미결:" 로 표기한다. 검증 없이 확정으로 적지 않는다.*